In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  yappu-fsm · single-cell Colab runner  (Runtime ▸ Run all, or Shift+Enter)
#  Tamil prosody FSM · constrained decoding for HF models · ICD-10-CM (US) linker
#  Everything is embedded below: no upload, no git clone, no model download.
# ════════════════════════════════════════════════════════════════════════════
RUN_TESTS = True          # full pytest suite (~1 min on CPU)
RUN_SERVER_DEMO = True    # OpenAI-compatible /v1/chat/completions on localhost:8000
CODING_SETTING = "outpatient"   # or "inpatient"  (ICD-10-CM Official Guidelines IV.H vs II.H)

FILES = {
    'pyproject.toml': '[build-system]\nrequires = ["setuptools>=68", "wheel"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "yappu-fsm"\nversion = "0.2.0"\ndescription = "Tamil prosody (Yappu Ilakkanam) as finite-state machines, constrained decoding for Hugging Face LLMs, and a rule-based clinical ICD-10 linker"\nreadme = "README.md"\nrequires-python = ">=3.9"\nlicense = {text = "MIT"}\ndependencies = []\n\n[project.optional-dependencies]\nhf = ["torch>=2.0", "transformers>=4.40", "tokenizers>=0.15"]\ndev = ["pytest>=7.4"]\n\n[project.scripts]\nyappu = "yappu_fsm.cli:main"\n\n[tool.setuptools.packages.find]\ninclude = ["yappu_fsm*"]\n\n[tool.setuptools.package-data]\nyappu_fsm = ["data/*.json", "data/*.jsonl", "data/*.ace"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\npythonpath = ["tests"]\nfilterwarnings = ["ignore::UserWarning", "ignore::FutureWarning"]\n',
    'README.md': '# yappu-fsm\n\nTamil prosody (**Yappu Ilakkanam**, யாப்பிலக்கணம்) as finite-state machines, constrained\ndecoding for Hugging Face LLMs, and a rule-based clinical entity linker for ICD-10 coding.\n\nThis repository consolidates and corrects all the code from the "Tamil Prosody → FSM →\nconstrained decoding → clinical ICD-10 coding" exploration into one tested package.\n\n| Module | What it does |\n|---|---|\n| `yappu_fsm.tamil` | Graphemes, Kuril/Nedil/Otru classes, Asai encoding (`1` = Ner, `0` = Nirai), Ethukai and Monai keys |\n| `yappu_fsm.prosody` | Seer table, Thalai naming, full **Nerisai Venba validator** (15 feet, Vendalai, Eetrucheer incl. Kaasu/Pirappu, Oru/Iru Vikarpam Ethukai) |\n| `yappu_fsm.lexicon` | Annotated word lexicon with dead-end-free lookahead |\n| `yappu_fsm.decoding` | Byte-trie state machines + `VenbaLogitsProcessor` and `ICD10LogitsProcessor` for `model.generate()` |\n| `yappu_fsm.clinical` | Abbreviations, clause walls, NegEx-style assertion, closest-token laterality, phonetic/fuzzy matching, ICD-10 lookup, MCC evaluation |\n| `yappu_fsm.rules` | ICD-10-CM coding rules in ACE-style controlled English, compiled to stratified Datalog, with proof traces |\n| `yappu_fsm.server` | OpenAI-compatible `/v1/chat/completions` endpoint (stdlib only) |\n| `yappu_fsm.cli` | `yappu asai | validate | code [--rules] | rules | eval | serve | demo-decode` |\n\n## Install\n\n```bash\npip install -e ".[dev]"          # core: pure Python, no dependencies\npip install -e ".[hf,dev]"       # + torch / transformers for constrained decoding\npytest -q                        # 142 tests; HF tests auto-skip without torch\n```\n\nColab: `notebooks/yappu_fsm_colab_single_cell.ipynb` is one self-contained cell. It embeds the\nwhole package, so nothing needs uploading; run it to get the tests and every demo.\n`docs/tutorial.html` is a progressive, interactive tutorial with comics.\n\n## Prosody\n\n```bash\n$ yappu asai கூடினால் இலக்கணம் கொக்கு\nகூடினால்    கூ/டினால்    10   Koovilam    Vilacheer   ethukai=N:ட monai=C:க:U\nஇலக்கணம்    இலக்/கணம்    00   Karuvilam   Vilacheer   ethukai=K:ல monai=V:I\nகொக்கு      கொக்/கு      11   Thema       Maacheer    ethukai=K:க monai=C:க:U   (Kaasu as final foot)\n\n$ yappu validate "அடக்க முடையா ரறிவிலரென் றெண்ணிக் ... வாடி யிருக்குமாம் கொக்கு."\n...\nVerdict: VALID Nerisai Venba (Iru Vikarpam)\n```\n\n```python\nfrom yappu_fsm import validate_nerisai_venba\nrep = validate_nerisai_venba(poem_text)      # or a list of 15 words\nprint("\\n".join(rep.lines()))\n```\n\nEthukai uses the traditional rule by default: the 2nd letters share a consonant and the\n1st letters agree in length. `strict_ethukai=True` requires the exact same 2nd letter.\n\n## Constrained decoding (Hugging Face)\n\nLLM tokens are sub-word pieces, often partial UTF-8 bytes for Tamil, so the Asai of a single\ntoken is meaningless. Both processors map every vocabulary token to its raw bytes\n(byte-level BPE and SentencePiece are both handled) and walk a byte trie of allowed whole\nwords or codes. A token is unmasked only if the text stays on a completable path. Allowed\nsets are cached per decoder state.\n\n```python\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessorList\nfrom yappu_fsm import VenbaLexicon, format_venba\nfrom yappu_fsm.decoding import VenbaDecoder, VenbaLogitsProcessor\n\ntok = AutoTokenizer.from_pretrained("your/causal-lm")\nmodel = AutoModelForCausalLM.from_pretrained("your/causal-lm")\nlexicon = VenbaLexicon(my_tamil_word_list)          # bigger lexicon -> better poems\nproc = VenbaLogitsProcessor(tok, VenbaDecoder(lexicon, monai_boost=3.0))\n\nids = tok("வெண்பா:", return_tensors="pt")\nout = model.generate(**ids, max_new_tokens=400, do_sample=True,\n                     logits_processor=LogitsProcessorList([proc]), eos_token_id=tok.eos_token_id)\nstate = proc.state_for(out[0, ids["input_ids"].shape[1]:].tolist())\nprint(format_venba(proc.decoder.finished_words(state)))\n```\n\nHard constraints are applied as `-inf` masks: Vendalai, no Kanicheer or 4-asai feet, Ethukai\nat feet 4, 7 and 12, and a valid Eetrucheer. Monai is a soft boost, ×1.5 at the 3rd foot\n(Pozhippu Monai). The lexicon precomputes which words keep a full poem reachable, so\ngeneration never dead-ends.\n\nICD-10: put a marker at the end of the prompt and only allowed codes can follow:\n\n```python\nfrom yappu_fsm.decoding import ICD10LogitsProcessor, parse_codes\nallowed = linker.allowed_codes(linker.process(note))   # codes from the rule-based linker\nproc = ICD10LogitsProcessor(tok, allowed)               # "CODES: X; Y" then newline / EOS\n```\n\n`yappu demo-decode` runs both processors offline. It trains a byte-level BPE tokenizer on the\nlexicon and uses a **randomly initialised** tiny GPT-2, which shows that validity comes from\nthe processor, not the model.\n\n**What is guaranteed.** Every completed generation is in the allowed language. For Venba,\n`validate_nerisai_venba` returns VALID. For ICD-10, every code is in the allowed set and none\nrepeats. The test suite checks this by property tests over random decoder paths and through\n`model.generate()`. The processors do **not** guarantee that a code is clinically correct,\nonly that it came from the allowed set.\n\n## Clinical linker\n\n```bash\n$ yappu code "Pt has lt knee osteoarthritus and mild NPDR OD. Denies SOB, but has uncontrolled HTN. R/O CAP."\nknee osteoarthritus  -> knee osteoarthritis                        PHONETIC_FUZZY PRESENT   LEFT        M17.12   CODE\nmild nonproliferative diabetic retinopathy -> (same)               EXACT          PRESENT   RIGHT       E11.3291 CODE\nshortness of breath  -> shortness of breath                        EXACT          NEGATED   UNSPECIFIED -        DO_NOT_CODE\nhypertension         -> hypertension                               EXACT          PRESENT   UNSPECIFIED I10      CODE\npneumonia            -> pneumonia                                  EXACT          UNCERTAIN UNSPECIFIED -        DO_NOT_CODE\nallowed codes: M17.12, E11.3291, I10\n```\n\n* **Assertion (NegEx + ConText):**\n  * Pre-triggers ("denies", "no", "without", "ruled out", "absence of", "no history of", …)\n    scan forward, and post-triggers ("was ruled out", "not demonstrated", "never developed",\n    "absent", "cannot be ruled out", "unlikely", "resolved") scan backward. Both reach at most 6\n    tokens and stop at CONJ terminators ("aside from", "secondary to", "due to", "except", …).\n  * Pseudo-triggers ("no change", "gram negative", "not ruled out") cannot open a negation.\n  * ConText adds HISTORICAL, FAMILY_HISTORY and HYPOTHETICAL ("return if", "if the patient\n    develops", "watch for").\n  * Clause walls (`.`, `;`, `but`, `however`, …) split the note first; decimals like `7.5` do not.\n* **Laterality:** closest-token binding within 4 tokens, with each modifier used once.\n  Bilateral hip pain maps to both side codes, because no single bilateral code exists.\n* **Fuzzy matching:** compared word by word. A word matches exactly, or, if it is at least 5\n  letters, by phonetic skeleton or edit similarity ≥ 0.84. So "osteoarthritus" and "pnemonia"\n  match, but "hip rain" does not match "hip pain".\n* **Abbreviations:** ambiguous ones (CAP, OD, OS, OU, MI) are case-sensitive, so "knee cap"\n  is not pneumonia.\n\n### Evaluation (MCC)\n\n```bash\n$ yappu eval\nn_pairs              31\nassertion_mcc        0.9558\nlaterality_mcc       1.0\ncodable_mcc          0.9269\n```\n\nNegEx alignment, measured on `data/gold_negex_heldout.jsonl` (12 notes written after the\nrules were tuned on `gold_negex.jsonl`, scored once before any further change):\n\n| | assertion MCC | codable MCC |\n|---|---|---|\n| previous linker | 0.619 | 0.539 |\n| NegEx/ConText linker | 0.861 | 0.811 |\n\nThe two misses that run exposed ("cannot be ruled out" after the term; "which" wrongly used\nas a terminator) were then fixed, so that set is now a regression test, not a held-out one.\nThe next honest measurement needs real notes.\n\n`data/gold_assertions.jsonl` is a **small synthetic smoke-test set** of 20 notes written for\nregression testing, not a clinical validation. Missed and spurious entities count as their\nown classes, so recall errors lower the MCC. The remaining error ("chest pain is unlikely\nto be cardiac" → UNCERTAIN) is kept as a known limitation of rule-based scope.\n\n### ICD-10-CM (US) coding rules\n\nThe linker targets **ICD-10-CM** and follows the *ICD-10-CM Official Guidelines for Coding\nand Reporting* for uncertain diagnoses:\n\n| Setting | Guideline | "Probable / suspected / rule out X" |\n|---|---|---|\n| `outpatient` (default) | Section IV.H | not coded (`DO_NOT_CODE`); code documented signs and symptoms instead |\n| `inpatient` | Section II.H / III.C | coded as if established (assertion stays `UNCERTAIN` for audit) |\n\nNegated findings are never coded. History maps to Z86/Z87 personal-history codes and family\nhistory to Z80–Z84 where the lexicon provides them.\n\n```bash\nyappu code "Probable pneumonia, start antibiotics."                      # DO_NOT_CODE\nyappu code "Probable pneumonia, start antibiotics." --setting inpatient  # J18.9\n```\n\n`data/icd10cm_example_lexicon.json` holds 32 common conditions (diabetes, hypertension, AF,\nheart failure, CKD, COPD, asthma, pneumonia, OA, joint pain with laterality, carpal tunnel,\nhistory and family-history codes, and more). It is a demo subset. Extend it in the same\nformat (`{"entity": {"ASSERTION_LATERALITY": code | [codes]}}`) and verify codes against the\ncurrent CMS release. All output is decision support for a certified coder.\n\n## Coding rules in controlled English → Datalog\n\nConventions that relate facts to each other (combination codes, "code first / use additional",\nExcludes1, symptoms integral to a diagnosis, setting-dependent guidelines) are written as\nACE-style sentences in `data/icd10cm_rules.ace`:\n\n```text\nDM_CKD:  If the patient has a type-2-diabetes and the patient has a chronic-kidney-disease\n         then the code E11.22 is assigned and the code E11.22 precedes the code N18.9.\nHTN_CKD: If the patient has a hypertension and the patient has a chronic-kidney-disease\n         then the code I12.9 is assigned and the code I10 is not assigned and the code I12.9 precedes the code N18.9.\nEXCL_E10_E11: If the code E10.9 is assigned and the code E11.9 is assigned then the codes E10.9 and E11.9 conflict.\nINPT_UNCERTAIN: If the encounter is an inpatient-encounter and the patient has an uncertain diagnosis X\n         then the candidate code of X is assigned.\n```\n\n```bash\nyappu rules             # parse, show the deterministic paraphrase of every rule\nyappu rules --datalog   # the compiled program, ready to load into another Datalog engine\nyappu code "Pt with T2DM and CKD. Also HTN." --rules     # E11.22, I12.9, N18.9 with proofs\n```\n\nHow it works:\n\n1. The linker turns the note into facts: `has(patient, type-2-diabetes)`, `uncertain(patient, pneumonia)`,\n   `side(...)`, `candidate(patient, concept, code)`, `code(patient, code)` and `setting(patient, inpatient)`.\n2. Rules are parsed **once** by `yappu_fsm.rules.ace`. Content words must be lexicon concepts,\n   anything outside the grammar is rejected with the exact word position, and each rule gets a\n   bracketed paraphrase for review. Each rule compiles to Horn clauses.\n3. `yappu_fsm.rules.datalog` evaluates the program: semi-naive, with stratified negation\n   (`final(P, C) :- code(P, C), not blocked(P, C).`). Unsafe or unstratifiable rule sets are\n   rejected at load time, so evaluation always terminates.\n4. The result holds the final codes (sequenced by `precedes`), blocked codes, Excludes1\n   conflicts for coder review, and a proof tree per code that cites the rule sentence.\n\n**Sequencing.** Each final code starts at a default position. A code the linker produced keeps\nits finding\'s place. A rule-derived code takes the place of the code it replaced (E11.22 where\nE11.9 was, I12.9 where I10 was); otherwise it sits at the earliest finding its proof mentions.\n`precedes` facts are then applied as a stable topological order. House sequencing policy is\nitself written as rules, using code variables and code kinds (diagnosis, symptom R, factor Z,\nexternal cause V–Y):\n\n```text\nSEQ_DX_SYMPTOM: If a code C is assigned and the code C is a symptom code and a code D is assigned\n                and the code D is a diagnosis code then the code D precedes the code C.\n```\n\n`"Pt with T2DM and CKD. Also HTN. Probable CAP. Cough for three days."` → `E11.22, I12.9, N18.9, R05.9`.\nThis is a default order for review; principal-diagnosis selection (Section II) is not modelled.\n\nThe language is a coding-rule **subset in the spirit of ACE** and is parsed by this package; it\ndoes not call the Attempto Parsing Engine. It has no wildcards over code families (E11.9 needs\none blocking rule per complication code).\n\n### Performance\n\nMeasured on CPU, Python 3.13 (`tests/test_clinical.py::test_index_scales` guards this):\n\n| | ms / note | notes / s |\n|---|---|---|\n| linker, 32-entry demo lexicon | 0.14 | ~7,400 |\n| linker + 16 coding rules (Datalog) | 0.30 | ~3,300 |\n| linker, 100,000-entry synthetic lexicon | 0.42 | ~2,400 |\n\nThe linker finds candidate entries through hash indexes on each entry\'s first word: exact\nform, phonetic skeleton, and one-deletion keys, which cover one substitution, insertion,\ndeletion or swap. A full scan is still available (`ClinicalLinker(index=False)`), and a test\nchecks that both give identical results on the gold notes and 300 random typos. A first word\nmore than two edits away from the lexicon term is not retrieved by the index.\n\n## OpenAI-compatible endpoint\n\n```bash\nyappu serve --port 8000\n```\n\n```python\nfrom openai import OpenAI\nc = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")\nr = c.chat.completions.create(model="yappu-icd-linker",\n        messages=[{"role": "user", "content": "Mild NPDR OD, family history of HTN."}])\nprint(r.choices[0].message.content)    # JSON: findings + allowed_codes\n```\n\nModels: `yappu-icd-linker`, `yappu-icd-rules` (codes + proofs) and `yappu-venba-validator`. Pass `"setting": "inpatient"` in the\nrequest body (`extra_body` in the OpenAI SDK) for the inpatient rule. Streaming is not supported.\n\n## Corrections relative to the original thread\n\n* Asai: kuril + kuril is one Nirai (கூ/டினால், இலக்/கணம், தவழ்ந்/திட, ஏற்/றிடும், பார்ப்/பவர்);\n  NFC normalisation; aytham handled.\n* Kaasu/Pirappu final feet are accepted (kutriyalukaram), so the classical Nanneri verse validates.\n* Ethukai compares the consonant of the 2nd letter plus 1st-letter length; the old exact-letter\n  rule rejected valid poems.\n* Monai consonant pairs follow Ina Ezhuthu: ச/த, ஞ/ந, ம/வ (not த/ந, ச/ய).\n* `SEER_CATEGORIES` filtered tuples against a string, so Kanicheer was never excluded.\n* Crashes fixed: `range(input_ids.shape)`, `extract_ethukai_key(words)` with a list,\n  `letters` returned instead of `letters[1]`, and a missing operator in the laterality binder.\n* Logits processors work on whole words and codes via byte tries instead of single tokens.\n  The ICD processor no longer loops over the full vocabulary in Python on every step\n  (results are cached), and duplicate codes can no longer be emitted.\n* Clinical: nearest-trigger assertion (the old version marked "Denies SOB, history of MI" as\n  HISTORICAL for SOB), trigger scope limit, pseudo-triggers, "resolved" → HISTORICAL.\n  Corrected CM mappings: mild NPDR is E11.329x without macular edema; E11.319 has no\n  laterality; there is no bilateral M25.55x code.\n* Tests that indexed `results` instead of `results[0]`, and the "hip rain" guardrail test\n  that could not pass, are fixed.\n\n## Limitations\n\n* The English Asai encoder is a rough heuristic. Real English meter needs a pronouncing\n  dictionary.\n* Venba quality depends on the lexicon. The bundled one (~55 words) is for tests.\n* The clinical linker is rule-based and lexicon-driven. Pair it with a trained NER and\n  assertion model for real charts. Trigger scope is clause-based: in "Probable CAP with cough\n  and fever" the symptoms are also read as uncertain (see `test_known_limitation_*`).\n* No model is trained here, so Sharpness-Aware Minimization does not apply. A Lean 4\n  formalisation of the decoder guarantee is not included yet; the guarantee is currently\n  checked by property tests.\n\nMIT licensed.\n',
    'LICENSE': 'MIT License\n\nCopyright (c) 2026 Dhanesh\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n',
    'yappu_fsm/__init__.py': '"""yappu-fsm: Tamil prosody (Yappu Ilakkanam) as finite-state machines, constrained\ndecoding for Hugging Face models, and a rule-based clinical ICD-10 linker."""\n\nfrom importlib import resources as _res\n\nfrom .lexicon import VenbaLexicon\nfrom .prosody import format_venba, validate_nerisai_venba\nfrom .tamil import encode, encode_asai, ethukai_key, graphemes, monai_key, split_display\n\n__version__ = "0.2.0"\n\n\ndef data_path(name: str) -> str:\n    """Path to a bundled data file (e.g. \'icd10cm_example_lexicon.json\')."""\n    return str(_res.files(__package__) / "data" / name)\n\n\n__all__ = ["VenbaLexicon", "format_venba", "validate_nerisai_venba", "encode", "encode_asai",\n           "ethukai_key", "graphemes", "monai_key", "split_display", "data_path", "__version__"]\n',
    'yappu_fsm/cli.py': '"""Command-line interface: ``yappu <command> ...``"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\n\nfrom . import data_path\nfrom .prosody import SEER, eetrucheer_vaipadu, format_venba, validate_nerisai_venba\nfrom .tamil import encode, ethukai_key, monai_key, split_display\n\n\ndef cmd_asai(a):\n    for w in a.words:\n        code = encode(w)\n        vp, cat = SEER.get(code, (eetrucheer_vaipadu(w), "Orasai/final" if eetrucheer_vaipadu(w) else None))\n        print(f"{w:<16} {split_display(w):<18} {code:<4} {(vp or \'-\'):<14} {(cat or \'-\'):<12} "\n              f"ethukai={ethukai_key(w)} monai={monai_key(w)}")\n\n\ndef cmd_validate(a):\n    text = sys.stdin.read() if a.poem in (None, "-") else a.poem\n    rep = validate_nerisai_venba(text, strict_ethukai=a.strict)\n    print("\\n".join(rep.lines()))\n    return 0 if rep.valid else 1\n\n\ndef _linker(path, setting="outpatient"):\n    from .clinical import ClinicalLinker, load_lexicon\n    return ClinicalLinker(load_lexicon(path or data_path("icd10cm_example_lexicon.json")), setting=setting)\n\n\ndef _coder(lexicon, rules):\n    from .rules import RuleCoder\n    return RuleCoder(_linker(lexicon), data_path("icd10cm_rules.ace") if rules in (None, "default") else rules)\n\n\ndef cmd_code(a):\n    text = sys.stdin.read() if a.note in (None, "-") else a.note\n    if a.rules is not None:\n        res = _coder(a.lexicon, a.rules).code(text, a.setting)\n        if a.json:\n            print(json.dumps({"codes": res.codes, "blocked": list(res.blocked),\n                              "conflicts": [[x, y] for x, y, _ in res.conflicts], "proofs": res.proofs},\n                             ensure_ascii=False, indent=2))\n        else:\n            print(res.report())\n        return\n    L = _linker(a.lexicon, a.setting)\n    findings = L.process(text)\n    if a.json:\n        print(json.dumps([f.to_dict() for f in findings], ensure_ascii=False, indent=2))\n    else:\n        for f in findings:\n            print(f"{f.raw_mention:<28} -> {f.entity:<42} {f.match_type:<15} {f.assertion:<15} "\n                  f"{f.laterality:<11} {\',\'.join(f.codes) or \'-\':<18} {f.status}")\n        print("allowed codes:", ", ".join(L.allowed_codes(findings)) or "(none)")\n\n\ndef cmd_rules(a):\n    from .rules import compile_rules, parse_rules\n    from .clinical import load_lexicon\n    path = a.file or data_path("icd10cm_rules.ace")\n    lex = load_lexicon(a.lexicon or data_path("icd10cm_example_lexicon.json"))\n    rules = parse_rules(open(path, encoding="utf-8").read(), concepts=lex.keys())\n    prog = compile_rules(rules)\n    if a.datalog:\n        print(prog.to_text())\n        return\n    for r in rules:\n        print(f"{r.rid}: {r.paraphrase()}")\n    print(f"\\n{len(rules)} rules OK, {len(prog.rules)} Datalog clauses, {len(prog.strata())} strata")\n\n\ndef cmd_eval(a):\n    from .clinical import evaluate\n    r = evaluate(_linker(a.lexicon), a.gold or data_path("gold_assertions.jsonl"))\n    for k, v in r.items():\n        if k != "errors":\n            print(f"{k:<20} {v}")\n    for e in r["errors"]:\n        print("ERROR:", e["note"], "| gold:", e["gold"], "| pred:", e["pred"])\n\n\ndef cmd_serve(a):\n    from .server import serve\n    srv = serve(a.host, a.port, a.lexicon)\n    print(f"OpenAI-compatible endpoint on http://{a.host}:{a.port}/v1/chat/completions")\n    try:\n        srv.serve_forever()\n    except KeyboardInterrupt:\n        pass\n\n\ndef cmd_demo_decode(a):\n    from .demo import run_demo\n    run_demo(seed=a.seed)\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(prog="yappu", description="Tamil prosody FSM + clinical ICD-10 linker")\n    sub = p.add_subparsers(dest="cmd", required=True)\n    s = sub.add_parser("asai", help="show asai split / seer / ethukai / monai for words")\n    s.add_argument("words", nargs="+")\n    s.set_defaults(fn=cmd_asai)\n    s = sub.add_parser("validate", help="validate a Nerisai Venba (text arg or stdin)")\n    s.add_argument("poem", nargs="?")\n    s.add_argument("--strict", action="store_true", help="exact 2nd-letter Ethukai")\n    s.set_defaults(fn=cmd_validate)\n    s = sub.add_parser("code", help="extract entities and ICD-10 codes from a note")\n    s.add_argument("note", nargs="?")\n    s.add_argument("--lexicon")\n    s.add_argument("--json", action="store_true")\n    s.add_argument("--setting", choices=["outpatient", "inpatient"], default="outpatient",\n                   help="ICD-10-CM guideline for uncertain diagnoses (default: outpatient)")\n    s.add_argument("--rules", nargs="?", const="default",\n                   help="apply controlled-English coding rules (default rule set, or a .ace file)")\n    s.set_defaults(fn=cmd_code)\n    s = sub.add_parser("rules", help="check a controlled-English rules file: paraphrases or Datalog")\n    s.add_argument("file", nargs="?")\n    s.add_argument("--lexicon")\n    s.add_argument("--datalog", action="store_true", help="print the compiled Datalog program")\n    s.set_defaults(fn=cmd_rules)\n    s = sub.add_parser("eval", help="MCC evaluation on a gold JSONL file")\n    s.add_argument("--gold")\n    s.add_argument("--lexicon")\n    s.set_defaults(fn=cmd_eval)\n    s = sub.add_parser("serve", help="run the OpenAI-compatible /v1/chat/completions server")\n    s.add_argument("--host", default="127.0.0.1")\n    s.add_argument("--port", type=int, default=8000)\n    s.add_argument("--lexicon")\n    s.set_defaults(fn=cmd_serve)\n    s = sub.add_parser("demo-decode", help="offline constrained-decoding demo (needs torch + transformers)")\n    s.add_argument("--seed", type=int, default=0)\n    s.set_defaults(fn=cmd_demo_decode)\n    a = p.parse_args(argv)\n    return a.fn(a) or 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n',
    'yappu_fsm/clinical/__init__.py': 'from .linker import ClinicalLinker, Finding, load_lexicon\nfrom .metrics import evaluate, mcc\nfrom .phonetic import clinical_phonetic_key, similarity\n\n__all__ = ["ClinicalLinker", "Finding", "load_lexicon", "evaluate", "mcc", "clinical_phonetic_key", "similarity"]\n',
    'yappu_fsm/clinical/linker.py': '"""Rule-based clinical entity recognition and ICD-10 linking.\n\nPipeline for each note:\n\n1. expand abbreviations (case-sensitive for ambiguous ones such as CAP, OD, OS, MI);\n2. split into clauses at boundary "walls" (., ;, newline, but, however, ...) –\n   the Sandhi analogy: a wall stops assertion scope from leaking;\n3. find lexicon entities by exact or fuzzy (phonetic / edit-distance) match;\n4. assign each entity the assertion of the **nearest preceding** trigger in its\n   clause (NEGATED / UNCERTAIN / HISTORICAL / FAMILY_HISTORY), or a post-trigger\n   right after it; pseudo-triggers ("no change", "gram negative") are ignored;\n5. bind laterality (RIGHT / LEFT / BILATERAL) by closest-token distance;\n6. look the (assertion, laterality) key up in the lexicon to get code(s).\n\nCoding setting follows the ICD-10-CM Official Guidelines for Coding and Reporting:\n\n* ``setting="outpatient"`` (default, Section IV.H): uncertain diagnoses ("probable",\n  "suspected", "rule out", ...) are NOT coded; code the documented signs/symptoms instead.\n* ``setting="inpatient"`` (Section II.H / III.C): an uncertain diagnosis documented at\n  discharge is coded as if it existed or was established.\n\nNegated findings are never coded. This is decision support only: output must be\nreviewed by a qualified coder.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom collections import defaultdict\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence, Tuple, Union\n\nfrom .phonetic import clinical_phonetic_key, phonetic_word, similarity\n\nCodeValue = Union[str, List[str]]\nLexicon = Dict[str, Dict[str, CodeValue]]\n\nSETTINGS = ("outpatient", "inpatient")\n\n\n@dataclass\nclass Finding:\n    raw_mention: str\n    entity: str\n    match_type: str\n    confidence: float\n    assertion: str\n    laterality: str\n    laterality_distance: Optional[int]\n    codes: List[str]\n    status: str               # CODE | DO_NOT_CODE | REVIEW_SPECIFICITY\n    clause: str\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\ndef load_lexicon(path: Union[str, Path]) -> Lexicon:\n    data = json.loads(Path(path).read_text(encoding="utf-8"))\n    return {k: v for k, v in data.items() if not k.startswith("_")}\n\n\nclass ClinicalLinker:\n    # (pattern, replacement, case_sensitive)\n    ABBREVIATIONS: List[Tuple[str, str, bool]] = [\n        (r"\\bt2dm\\b", "type 2 diabetes", False),\n        (r"\\bhtn\\b", "hypertension", False),\n        (r"\\bsob\\b", "shortness of breath", False),\n        (r"\\bnpdr\\b", "nonproliferative diabetic retinopathy", False),\n        (r"\\bb/l\\b", "bilateral", False),\n        (r"\\ba-?fib\\b", "atrial fibrillation", False),\n        (r"\\bchf\\b", "heart failure", False),\n        (r"\\bckd\\b", "chronic kidney disease", False),\n        (r"\\buti\\b", "urinary tract infection", False),\n        (r"\\bhld\\b", "hyperlipidemia", False),\n        (r"\\bcva\\b", "stroke", False),\n        (r"\\bcts\\b", "carpal tunnel syndrome", False),\n        (r"\\bt1dm\\b", "type 1 diabetes", False),\n        (r"\\brt\\b", "right", False),\n        (r"\\blt\\b", "left", False),\n        (r"\\bMI\\b", "myocardial infarction", True),\n        (r"\\bCAP\\b", "pneumonia", True),\n        (r"\\bOD\\b", "right eye", True),\n        (r"\\bOS\\b", "left eye", True),\n        (r"\\bOU\\b", "bilateral eyes", True),\n    ]\n\n    LATERALITY: Dict[str, List[str]] = {\n        "BILATERAL": [r"\\bbilateral eyes\\b", r"\\bboth eyes\\b", r"\\bbilateral\\b", r"\\bboth\\b"],\n        "RIGHT": [r"\\bright eye\\b", r"\\bright-sided\\b", r"\\bright\\b"],\n        "LEFT": [r"\\bleft eye\\b", r"\\bleft-sided\\b", r"\\bleft\\b"],\n    }\n\n    # Trigger lists follow NegEx (Chapman et al. 2001) and its ConText extension\n    # (Harkema et al. 2009): pre-triggers scan forward, post-triggers scan backward, both\n    # over at most ``max_scope_tokens`` tokens and never across a CONJ terminator.\n    PRE_TRIGGERS: Dict[str, List[str]] = {\n        "NEGATED": [r"\\bdenies\\b", r"\\bdenied\\b", r"\\bno evidence of\\b", r"\\bnegative for\\b",\n                    r"\\bwithout\\b", r"\\bno\\b", r"\\bfree of\\b", r"\\bruled out\\b", r"\\brules out\\b",\n                    r"\\babsence of\\b", r"\\bno signs? of\\b", r"\\bno history of\\b", r"\\bnot\\b",\n                    r"\\bnever had\\b", r"\\bnone\\b"],\n        "UNCERTAIN": [r"\\brule out\\b", r"\\br/o\\b", r"\\bpossible\\b", r"\\bsuspected\\b",\n                      r"\\bprobable\\b", r"\\bquestionable\\b", r"\\bconcern for\\b",\n                      r"\\bnot (?:been )?ruled out\\b", r"\\bcannot be ruled out\\b", r"\\bnot rule out\\b"],\n        "HISTORICAL": [r"\\bhistory of\\b", r"\\bh/o\\b", r"\\bpmh\\b", r"\\bstatus post\\b", r"\\bs/p\\b",\n                       r"\\bprevious\\b", r"\\bprior\\b"],\n        "FAMILY_HISTORY": [r"\\bfamily history of\\b", r"\\bfhx\\b", r"\\bmother had\\b", r"\\bfather had\\b",\n                           r"\\bmother has\\b", r"\\bfather has\\b", r"\\bsister had\\b", r"\\bbrother had\\b"],\n        "HYPOTHETICAL": [r"\\bif (?:the patient |he |she |you |they )?develops?\\b", r"\\breturn if\\b",\n                         r"\\bcall if\\b", r"\\bshould (?:the patient |he |she |you |they )?develop\\b",\n                         r"\\bin case of\\b", r"\\bwatch for\\b"],\n    }\n    POST_TRIGGERS: Dict[str, List[str]] = {\n        "NEGATED": [r"\\b(?:was |is |has been |were )?ruled out\\b", r"\\b(?:is |was )?not present\\b",\n                    r"\\bnot demonstrated\\b", r"\\bnot seen\\b", r"\\bnever developed\\b", r"\\babsent\\b"],\n        "HISTORICAL": [r"\\b(?:has |is |was )?resolved\\b"],\n        "UNCERTAIN": [r"\\b(?:is |was )?(?:suspected|unlikely|possible|questionable)\\b",\n                      r"\\bcannot be (?:ruled out|excluded)\\b", r"\\bnot (?:been )?ruled out\\b"],\n    }\n    # Pseudo-triggers stop a NEGATED trigger from opening a scope (NegEx [PSEU]).\n    PSEUDO_TRIGGERS = [r"\\bno change\\b", r"\\bno increase\\b", r"\\bnot only\\b", r"\\bgram negative\\b",\n                       r"\\bwithout difficulty\\b", r"\\bnot (?:been )?ruled out\\b", r"\\bnot rule out\\b",\n                       r"\\bcannot be ruled out\\b", r"\\bno further\\b", r"\\bnot certain whether\\b"]\n    # CONJ terminators end a scope inside a clause (NegEx [CONJ]).\n    CONJ = [r"\\baside from\\b", r"\\bapart from\\b", r"\\bsecondary to\\b", r"\\bcause of\\b", r"\\bsource of\\b",\n            r"\\bdue to\\b", r"\\bexcept\\b"]\n    # Clause walls (also CONJ terminators) split the note before any matching.\n    WALLS = [r"\\bbut\\b", r"\\bhowever\\b", r"\\bexcept\\b", r"\\balthough\\b", r"\\bwhereas\\b",\n             r"\\.(?!\\d)", r";", r"\\n"]\n\n    def __init__(self, lexicon: Lexicon, max_laterality_distance: int = 4,\n                 fuzzy_threshold: float = 0.84, min_fuzzy_word_len: int = 5,\n                 post_window_chars: int = 30, max_scope_tokens: int = 6,\n                 setting: str = "outpatient", index: bool = True):\n        if setting not in SETTINGS:\n            raise ValueError(f"setting must be one of {SETTINGS}")\n        self.setting = setting\n        self.lexicon = {k.lower(): v for k, v in lexicon.items()}\n        self.max_lat = max_laterality_distance\n        self.fuzzy_threshold = fuzzy_threshold\n        self.min_fuzzy_len = min_fuzzy_word_len\n        self.post_window = post_window_chars\n        self.max_scope = max_scope_tokens\n        self.index = [{"canonical": k, "words": k.split(), "code_map": v} for k, v in self.lexicon.items()]\n        self.max_ngram = max((len(e["words"]) for e in self.index), default=1)\n        self.use_index = index\n        self._first_word_index = defaultdict(set)\n        self._cand_cache: Dict[str, Tuple[int, ...]] = {}\n        if index:\n            for i, e in enumerate(self.index):\n                for key in self._index_keys(e["words"][0]):\n                    self._first_word_index[key].add(i)\n\n    # ------------------------------------------------------------------ index\n    def _index_keys(self, w: str):\n        """Hash keys for candidate lookup: exact word, phonetic skeleton and\n        SymSpell-style one-deletion variants (symmetric, so they cover a single\n        substitution, insertion, deletion or adjacent transposition)."""\n        yield "x:" + w\n        if len(w) >= self.min_fuzzy_len:\n            yield "p:" + phonetic_word(w)\n            yield "d:" + w\n            for i in range(len(w)):\n                yield "d:" + w[:i] + w[i + 1:]\n\n    def _candidates(self, first_word: str) -> Sequence[int]:\n        """Indices of lexicon entries whose first word could match ``first_word``."""\n        if not self.use_index:\n            return range(len(self.index))\n        hit = self._cand_cache.get(first_word)\n        if hit is None:\n            found = set()\n            for key in self._index_keys(first_word):\n                found |= self._first_word_index.get(key, set())\n            hit = tuple(sorted(found))\n            if len(self._cand_cache) < 100_000:\n                self._cand_cache[first_word] = hit\n        return hit\n\n    # ------------------------------------------------------------------ text\n    def expand_abbreviations(self, text: str) -> str:\n        for pat, rep, cs in self.ABBREVIATIONS:\n            text = re.sub(pat, rep, text, flags=0 if cs else re.IGNORECASE)\n        return text\n\n    def split_clauses(self, text: str) -> List[str]:\n        parts = re.split("|".join(self.WALLS), text, flags=re.IGNORECASE)\n        return [p.strip() for p in parts if p and p.strip()]\n\n    @staticmethod\n    def _tokens(text: str) -> List[Tuple[str, int, int]]:\n        return [(m.group(0), m.start(), m.end()) for m in re.finditer(r"[\\w/\'-]+", text)]\n\n    @staticmethod\n    def _char_to_tok(c0: int, c1: int, toks) -> Tuple[int, int]:\n        s = next((i for i, t in enumerate(toks) if t[2] > c0), len(toks) - 1)\n        e = max((i for i, t in enumerate(toks) if t[1] < c1), default=s)\n        return s, e\n\n    # --------------------------------------------------------------- matching\n    def _word_match(self, a: str, b: str) -> Optional[str]:\n        if a == b:\n            return "EXACT"\n        if len(a) < self.min_fuzzy_len or len(b) < self.min_fuzzy_len:\n            return None\n        if phonetic_word(a) == phonetic_word(b) and similarity(a, b) >= 0.75:\n            return "PHONETIC"\n        if similarity(a, b) >= self.fuzzy_threshold:\n            return "EDIT"\n        return None\n\n    def match_entities(self, toks) -> List[dict]:\n        words = [t[0].lower() for t in toks]\n        taken, found = set(), []\n        for n in range(self.max_ngram, 0, -1):\n            for i in range(len(words) - n + 1):\n                if any(j in taken for j in range(i, i + n)):\n                    continue\n                span = words[i:i + n]\n                best = None\n                for ci in self._candidates(words[i]):\n                    e = self.index[ci]\n                    if len(e["words"]) != n:\n                        continue\n                    kinds = [self._word_match(a, b) for a, b in zip(span, e["words"])]\n                    if None in kinds:\n                        continue\n                    phrase, canon = " ".join(span), e["canonical"]\n                    if all(k == "EXACT" for k in kinds):\n                        mt, score = "EXACT", 1.0\n                    else:\n                        mt = "PHONETIC_FUZZY" if "EDIT" not in kinds else "EDIT_FUZZY"\n                        score = similarity(phrase, canon)\n                    if best is None or score > best[1]:\n                        best = (e, score, mt)\n                if best:\n                    e, score, mt = best\n                    taken.update(range(i, i + n))\n                    found.append({"raw": " ".join(t[0] for t in toks[i:i + n]), "entry": e,\n                                  "score": round(score, 3), "match_type": mt,\n                                  "c0": toks[i][1], "c1": toks[i + n - 1][2], "t0": i, "t1": i + n - 1})\n        return sorted(found, key=lambda x: x["t0"])\n\n    # -------------------------------------------------------------- assertion\n    def _triggers(self, clause_l: str) -> List[Tuple[int, int, str]]:\n        pseudo = [(m.start(), m.end()) for p in self.PSEUDO_TRIGGERS for m in re.finditer(p, clause_l)]\n        hits = []\n        for state, pats in self.PRE_TRIGGERS.items():\n            for p in pats:\n                for m in re.finditer(p, clause_l):\n                    if state == "NEGATED" and any(a <= m.start() < b for a, b in pseudo):\n                        continue\n                    hits.append((m.start(), m.end(), state))\n        # keep the longest of overlapping triggers ("family history of" beats "history of")\n        kept: List[Tuple[int, int, str]] = []\n        for h in sorted(hits, key=lambda h: -(h[1] - h[0])):\n            if all(h[1] <= k[0] or h[0] >= k[1] for k in kept):\n                kept.append(h)\n        return sorted(kept)\n\n    def _ntok(self, text: str) -> int:\n        return len(re.findall(r"[\\w/\'-]+", text))\n\n    def _conj_between(self, clause_l: str, a: int, b: int) -> bool:\n        seg = clause_l[a:b]\n        return any(re.search(p, seg) for p in self.CONJ)\n\n    def _post_hits(self, clause_l: str) -> List[Tuple[int, int, str]]:\n        pseudo = [(m.start(), m.end()) for p in self.PSEUDO_TRIGGERS for m in re.finditer(p, clause_l)]\n        hits = []\n        for state, pats in self.POST_TRIGGERS.items():\n            for p in pats:\n                for m in re.finditer(p, clause_l):\n                    if state == "NEGATED" and any(a <= m.start() < b for a, b in pseudo):\n                        continue\n                    hits.append((m.start(), m.end(), state))\n        return hits\n\n    def _assertion(self, clause_l: str, triggers, c0: int, c1: int) -> str:\n        """NegEx/ConText scope: the nearest pre-trigger before the entity, within the token\n        window and with no CONJ terminator in between; a post-trigger after the entity under\n        the same conditions overrides it."""\n        state = "PRESENT"\n        pre = [t for t in triggers if t[1] <= c0 and self._ntok(clause_l[t[1]:c0]) <= self.max_scope\n               and not self._conj_between(clause_l, t[1], c0)]\n        if pre:\n            state = max(pre, key=lambda t: t[1])[2]\n        post = [t for t in self._post_hits(clause_l) if t[0] >= c1 and self._ntok(clause_l[c1:t[0]]) <= self.max_scope\n                and not self._conj_between(clause_l, c1, t[0])]\n        if post:\n            state = min(post, key=lambda t: t[0])[2]\n        return state\n\n    # ------------------------------------------------------------- laterality\n    def _laterality_mentions(self, clause_l: str, toks) -> List[dict]:\n        out = []\n        for label, pats in self.LATERALITY.items():\n            for p in pats:\n                for m in re.finditer(p, clause_l):\n                    t0, t1 = self._char_to_tok(m.start(), m.end(), toks)\n                    if any(not (t1 < o["t0"] or t0 > o["t1"]) for o in out):\n                        continue\n                    out.append({"label": label, "t0": t0, "t1": t1, "used": False})\n        return out\n\n    def _bind_laterality(self, t0: int, t1: int, mentions) -> Tuple[str, Optional[int]]:\n        best, best_d = None, None\n        for m in mentions:\n            if m["used"]:\n                continue\n            d = t0 - m["t1"] if m["t1"] < t0 else (m["t0"] - t1 if m["t0"] > t1 else 0)\n            if d <= self.max_lat and (best_d is None or d < best_d):\n                best, best_d = m, d\n        if best is None:\n            return "UNSPECIFIED", None\n        best["used"] = True\n        return best["label"], best_d\n\n    # ---------------------------------------------------------------- public\n    def process(self, note: str) -> List[Finding]:\n        findings: List[Finding] = []\n        for clause in self.split_clauses(self.expand_abbreviations(note)):\n            cl = clause.lower()\n            toks = self._tokens(cl)\n            if not toks:\n                continue\n            triggers = self._triggers(cl)\n            lats = self._laterality_mentions(cl, toks)\n            for ent in self.match_entities(toks):\n                assertion = self._assertion(cl, triggers, ent["c0"], ent["c1"])\n                lat, dist = self._bind_laterality(ent["t0"], ent["t1"], lats)\n                cmap = ent["entry"]["code_map"]\n                lookup = assertion\n                if assertion == "UNCERTAIN" and self.setting == "inpatient":\n                    lookup = "PRESENT"   # CM Section II.H: code as if established\n                if lookup in ("NEGATED", "UNCERTAIN", "HYPOTHETICAL"):\n                    codes, status = [], "DO_NOT_CODE"\n                else:\n                    val = cmap.get(f"{lookup}_{lat}") or cmap.get(f"{lookup}_UNSPECIFIED") or cmap.get(lookup)\n                    codes = [] if val is None else ([val] if isinstance(val, str) else list(val))\n                    status = "CODE" if codes else "REVIEW_SPECIFICITY"\n                findings.append(Finding(ent["raw"], ent["entry"]["canonical"], ent["match_type"], ent["score"],\n                                        assertion, lat, dist, codes, status, clause))\n        return findings\n\n    @staticmethod\n    def allowed_codes(findings: Sequence[Finding]) -> List[str]:\n        """Codes the ICD-10 constrained decoder may emit for this note."""\n        out: List[str] = []\n        for f in findings:\n            for c in f.codes:\n                if c not in out:\n                    out.append(c)\n        return out\n',
    'yappu_fsm/clinical/metrics.py': '"""Matthews Correlation Coefficient (binary and multiclass, Gorodkin R_K) and the\nassertion / coding evaluation used by ``yappu eval``."""\n\nfrom __future__ import annotations\n\nimport json\nimport math\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Sequence, Tuple, Union\n\nMISSED = "MISSED"      # gold entity the system did not find\nSPURIOUS = "NONE"      # system entity with no gold counterpart\n\n\ndef mcc(y_true: Sequence[str], y_pred: Sequence[str]) -> float:\n    """Multiclass MCC (Gorodkin 2004); equals the usual binary MCC for 2 classes.\n\n    Returns 0.0 when undefined (e.g. a single class only)."""\n    if len(y_true) != len(y_pred):\n        raise ValueError("length mismatch")\n    s = len(y_true)\n    if s == 0:\n        return 0.0\n    c = sum(t == p for t, p in zip(y_true, y_pred))\n    t_k, p_k = Counter(y_true), Counter(y_pred)\n    labels = set(t_k) | set(p_k)\n    num = c * s - sum(t_k[k] * p_k[k] for k in labels)\n    den = math.sqrt((s * s - sum(p_k[k] ** 2 for k in labels)) * (s * s - sum(t_k[k] ** 2 for k in labels)))\n    return 0.0 if den == 0 else num / den\n\n\ndef confusion(y_true, y_pred) -> Dict[Tuple[str, str], int]:\n    return dict(Counter(zip(y_true, y_pred)))\n\n\ndef align(gold: List[Tuple[str, str, str]], pred: List[Tuple[str, str, str]]):\n    """Pair gold and predicted (entity, assertion, laterality) by entity name, in order."""\n    pool = list(pred)\n    pairs = []\n    for g in gold:\n        idx = next((i for i, p in enumerate(pool) if p[0] == g[0]), None)\n        if idx is None:\n            pairs.append((g, None))\n        else:\n            pairs.append((g, pool.pop(idx)))\n    pairs += [(None, p) for p in pool]\n    return pairs\n\n\ndef evaluate(linker, gold_path: Union[str, Path]) -> dict:\n    """Run the linker over a JSONL gold file and report MCC.\n\n    * ``assertion_mcc``  – multiclass MCC over assertion labels (missed / spurious\n      entities count as their own classes, so recall errors are penalised);\n    * ``laterality_mcc`` – multiclass MCC over laterality labels for found entities;\n    * ``codable_mcc``    – binary MCC of "should this mention produce a code?"\n      (PRESENT / HISTORICAL / FAMILY_HISTORY vs NEGATED / UNCERTAIN / missed).\n    """\n    a_t, a_p, l_t, l_p, b_t, b_p = [], [], [], [], [], []\n    errors = []\n    for line in Path(gold_path).read_text(encoding="utf-8").splitlines():\n        if not line.strip():\n            continue\n        row = json.loads(line)\n        gold = [tuple(g) for g in row["gold"]]\n        pred = [(f.entity, f.assertion, f.laterality) for f in linker.process(row["note"])]\n        for g, p in align(gold, pred):\n            gt = g[1] if g else SPURIOUS\n            pt = p[1] if p else MISSED\n            a_t.append(gt)\n            a_p.append(pt)\n            codable = lambda x: x in ("PRESENT", "HISTORICAL", "FAMILY_HISTORY")\n            b_t.append(str(codable(gt)))\n            b_p.append(str(codable(pt)))\n            if g and p:\n                l_t.append(g[2])\n                l_p.append(p[2])\n            if g != p:\n                errors.append({"note": row["note"], "gold": g, "pred": p})\n    return {\n        "n_pairs": len(a_t),\n        "assertion_mcc": round(mcc(a_t, a_p), 4),\n        "laterality_mcc": round(mcc(l_t, l_p), 4),\n        "codable_mcc": round(mcc(b_t, b_p), 4),\n        "assertion_accuracy": round(sum(t == p for t, p in zip(a_t, a_p)) / max(1, len(a_t)), 4),\n        "errors": errors,\n    }\n',
    'yappu_fsm/clinical/phonetic.py': '"""Phonetic skeleton keys and fuzzy similarity for clinical terms.\n\nThe skeleton plays the role of Monai\'s Ina Ezhuthu equivalence classes:\nspelling variants that sound alike (ph/f, pn-/n-, c/k/q, z/s, doubled letters,\ndropped interior vowels) collapse to the same key.\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom difflib import SequenceMatcher\n\n\ndef phonetic_word(w: str) -> str:\n    w = w.lower()\n    if len(w) <= 3:\n        return w.upper()\n    w = re.sub(r"^pn", "n", w)\n    w = re.sub(r"^kn", "n", w)\n    w = re.sub(r"^ps", "s", w)\n    w = w.replace("ph", "f").replace("gh", "g").replace("ck", "k")\n    w = re.sub(r"[cq]", "k", w)\n    w = w.replace("z", "s").replace("y", "i")\n    w = re.sub(r"([a-z])\\1+", r"\\1", w)\n    return w[0].upper() + re.sub(r"[aeiou]", "", w[1:]).upper()\n\n\ndef clinical_phonetic_key(phrase: str) -> str:\n    """\'osteoarthritis\' and \'osteoarthritus\' -> \'OSTRTHRTS\'."""\n    return " ".join(phonetic_word(w) for w in re.findall(r"[a-zA-Z]+", phrase))\n\n\ndef similarity(a: str, b: str) -> float:\n    return SequenceMatcher(None, a.lower(), b.lower()).ratio()\n',
    'yappu_fsm/data/gold_assertions.jsonl': '{"note": "Patient denies chest pain, but has uncontrolled hypertension.", "gold": [["chest pain","NEGATED","UNSPECIFIED"],["hypertension","PRESENT","UNSPECIFIED"]]}\n{"note": "History of MI in 2021. Mother had type 2 diabetes.", "gold": [["myocardial infarction","HISTORICAL","UNSPECIFIED"],["type 2 diabetes","FAMILY_HISTORY","UNSPECIFIED"]]}\n{"note": "Rule out pneumonia.", "gold": [["pneumonia","UNCERTAIN","UNSPECIFIED"]]}\n{"note": "Pt has lt knee osteoarthritis and mild NPDR OD.", "gold": [["knee osteoarthritis","PRESENT","LEFT"],["mild nonproliferative diabetic retinopathy","PRESENT","RIGHT"]]}\n{"note": "Exam shows b/l knee osteoarthritis. Mild NPDR OU noted.", "gold": [["knee osteoarthritis","PRESENT","BILATERAL"],["mild nonproliferative diabetic retinopathy","PRESENT","BILATERAL"]]}\n{"note": "Denies SOB, history of MI.", "gold": [["shortness of breath","NEGATED","UNSPECIFIED"],["myocardial infarction","HISTORICAL","UNSPECIFIED"]]}\n{"note": "Chest X-ray clear; pneumonia ruled out.", "gold": [["pneumonia","NEGATED","UNSPECIFIED"]]}\n{"note": "No known drug allergies, takes lisinopril for longstanding hypertension.", "gold": [["hypertension","PRESENT","UNSPECIFIED"]]}\n{"note": "Right hip pain for two weeks. No chest pain.", "gold": [["hip pain","PRESENT","RIGHT"],["chest pain","NEGATED","UNSPECIFIED"]]}\n{"note": "Family history of HTN. Pt has T2DM.", "gold": [["hypertension","FAMILY_HISTORY","UNSPECIFIED"],["type 2 diabetes","PRESENT","UNSPECIFIED"]]}\n{"note": "Possible CAP on imaging, start antibiotics.", "gold": [["pneumonia","UNCERTAIN","UNSPECIFIED"]]}\n{"note": "Labs consistent with diabetic nefropathy.", "gold": [["diabetic nephropathy","PRESENT","UNSPECIFIED"]]}\n{"note": "Pt presents with uncontrolled hypertensn and pnemonia.", "gold": [["hypertension","PRESENT","UNSPECIFIED"],["pneumonia","PRESENT","UNSPECIFIED"]]}\n{"note": "Pneumonia has resolved after treatment.", "gold": [["pneumonia","HISTORICAL","UNSPECIFIED"]]}\n{"note": "Bilateral hip pain worse on stairs.", "gold": [["hip pain","PRESENT","BILATERAL"]]}\n{"note": "Without shortness of breath at rest; chest pain on exertion.", "gold": [["shortness of breath","NEGATED","UNSPECIFIED"],["chest pain","PRESENT","UNSPECIFIED"]]}\n{"note": "Suspected diabetic retinopathy, refer to ophthalmology.", "gold": [["diabetic retinopathy","UNCERTAIN","UNSPECIFIED"]]}\n{"note": "s/p MI 2019, now stable. Denies chest pain.", "gold": [["myocardial infarction","HISTORICAL","UNSPECIFIED"],["chest pain","NEGATED","UNSPECIFIED"]]}\n{"note": "The patient\'s father had hypertension.", "gold": [["hypertension","FAMILY_HISTORY","UNSPECIFIED"]]}\n{"note": "Chest pain is unlikely to be cardiac; knee osteoarthritis right side.", "gold": [["chest pain","PRESENT","UNSPECIFIED"],["knee osteoarthritis","PRESENT","RIGHT"]]}\n',
    'yappu_fsm/data/gold_negex.jsonl': '{"note": "Ruled out pneumonia on admission.", "gold": [["pneumonia","NEGATED","UNSPECIFIED"]]}\n{"note": "Pneumonia in the left lower lobe was ruled out.", "gold": [["pneumonia","NEGATED","LEFT"]]}\n{"note": "No history of MI.", "gold": [["myocardial infarction","NEGATED","UNSPECIFIED"]]}\n{"note": "No fever aside from mild cough.", "gold": [["fever","NEGATED","UNSPECIFIED"],["cough","PRESENT","UNSPECIFIED"]]}\n{"note": "Absence of chest pain at rest.", "gold": [["chest pain","NEGATED","UNSPECIFIED"]]}\n{"note": "Headache was not demonstrated on review.", "gold": [["headache","NEGATED","UNSPECIFIED"]]}\n{"note": "Return if fever develops.", "gold": [["fever","HYPOTHETICAL","UNSPECIFIED"]]}\n{"note": "If the patient develops chest pain, call 911.", "gold": [["chest pain","HYPOTHETICAL","UNSPECIFIED"]]}\n{"note": "Cough secondary to pneumonia, no headache.", "gold": [["cough","PRESENT","UNSPECIFIED"],["pneumonia","PRESENT","UNSPECIFIED"],["headache","NEGATED","UNSPECIFIED"]]}\n{"note": "No signs of urinary tract infection.", "gold": [["urinary tract infection","NEGATED","UNSPECIFIED"]]}\n{"note": "Not rule out pneumonia yet, start empiric therapy.", "gold": [["pneumonia","UNCERTAIN","UNSPECIFIED"]]}\n{"note": "Fever never developed after surgery.", "gold": [["fever","NEGATED","UNSPECIFIED"]]}\n',
    'yappu_fsm/data/gold_negex_heldout.jsonl': '{"note": "Patient is free of chest pain today and has no cough.", "gold": [["chest pain","NEGATED","UNSPECIFIED"],["cough","NEGATED","UNSPECIFIED"]]}\n{"note": "Atrial fibrillation not seen on telemetry.", "gold": [["atrial fibrillation","NEGATED","UNSPECIFIED"]]}\n{"note": "Instructed to watch for fever and return if headache worsens.", "gold": [["fever","HYPOTHETICAL","UNSPECIFIED"],["headache","HYPOTHETICAL","UNSPECIFIED"]]}\n{"note": "Pneumonia cannot be ruled out.", "gold": [["pneumonia","UNCERTAIN","UNSPECIFIED"]]}\n{"note": "Denies headache, fever or cough.", "gold": [["headache","NEGATED","UNSPECIFIED"],["fever","NEGATED","UNSPECIFIED"],["cough","NEGATED","UNSPECIFIED"]]}\n{"note": "Cough present; fever absent.", "gold": [["cough","PRESENT","UNSPECIFIED"],["fever","NEGATED","UNSPECIFIED"]]}\n{"note": "No acute distress. Known hypertension on two agents.", "gold": [["hypertension","PRESENT","UNSPECIFIED"]]}\n{"note": "Shortness of breath due to heart failure, without chest pain.", "gold": [["shortness of breath","PRESENT","UNSPECIFIED"],["heart failure","PRESENT","UNSPECIFIED"],["chest pain","NEGATED","UNSPECIFIED"]]}\n{"note": "The patient was evaluated for stroke, which was ruled out.", "gold": [["stroke","NEGATED","UNSPECIFIED"]]}\n{"note": "Sister had breast pain; mother had colon cancer.", "gold": [["colon cancer","FAMILY_HISTORY","UNSPECIFIED"]]}\n{"note": "No longer smokes; history of nicotine dependence.", "gold": [["nicotine dependence","HISTORICAL","UNSPECIFIED"]]}\n{"note": "Asthma well controlled, no wheeze or cough today.", "gold": [["asthma","PRESENT","UNSPECIFIED"],["cough","NEGATED","UNSPECIFIED"]]}\n',
    'yappu_fsm/data/icd10cm_example_lexicon.json': '{\n  "_comment": "ICD-10-CM (US) demo lexicon, FY2025+ code set. Codes were checked against the CM Tabular List structure, but verify against the current official release before production use. E11.* codes presume documented type 2 diabetes. Format: {entity: {ASSERTION_LATERALITY: code | [codes]}}; UNSPECIFIED is the fallback laterality.",\n  "asthma": {\n    "PRESENT_UNSPECIFIED": "J45.909"\n  },\n  "atrial fibrillation": {\n    "PRESENT_UNSPECIFIED": "I48.91"\n  },\n  "carpal tunnel syndrome": {\n    "PRESENT_RIGHT": "G56.01",\n    "PRESENT_LEFT": "G56.02",\n    "PRESENT_BILATERAL": "G56.03",\n    "PRESENT_UNSPECIFIED": "G56.00"\n  },\n  "chest pain": {\n    "PRESENT_UNSPECIFIED": "R07.9"\n  },\n  "chronic kidney disease": {\n    "PRESENT_UNSPECIFIED": "N18.9"\n  },\n  "colon cancer": {\n    "PRESENT_UNSPECIFIED": "C18.9",\n    "HISTORICAL_UNSPECIFIED": "Z85.038",\n    "FAMILY_HISTORY_UNSPECIFIED": "Z80.0"\n  },\n  "copd": {\n    "PRESENT_UNSPECIFIED": "J44.9"\n  },\n  "cough": {\n    "PRESENT_UNSPECIFIED": "R05.9"\n  },\n  "depression": {\n    "PRESENT_UNSPECIFIED": "F32.A"\n  },\n  "diabetic nephropathy": {\n    "PRESENT_UNSPECIFIED": "E11.21"\n  },\n  "diabetic retinopathy": {\n    "PRESENT_UNSPECIFIED": "E11.319"\n  },\n  "fever": {\n    "PRESENT_UNSPECIFIED": "R50.9"\n  },\n  "gerd": {\n    "PRESENT_UNSPECIFIED": "K21.9"\n  },\n  "headache": {\n    "PRESENT_UNSPECIFIED": "R51.9"\n  },\n  "heart failure": {\n    "PRESENT_UNSPECIFIED": "I50.9"\n  },\n  "hip pain": {\n    "PRESENT_RIGHT": "M25.551",\n    "PRESENT_LEFT": "M25.552",\n    "PRESENT_BILATERAL": [\n      "M25.551",\n      "M25.552"\n    ],\n    "PRESENT_UNSPECIFIED": "M25.559"\n  },\n  "hyperlipidemia": {\n    "PRESENT_UNSPECIFIED": "E78.5"\n  },\n  "hypertension": {\n    "PRESENT_UNSPECIFIED": "I10",\n    "FAMILY_HISTORY_UNSPECIFIED": "Z82.49"\n  },\n  "knee osteoarthritis": {\n    "PRESENT_RIGHT": "M17.11",\n    "PRESENT_LEFT": "M17.12",\n    "PRESENT_BILATERAL": "M17.0",\n    "PRESENT_UNSPECIFIED": "M17.9"\n  },\n  "knee pain": {\n    "PRESENT_RIGHT": "M25.561",\n    "PRESENT_LEFT": "M25.562",\n    "PRESENT_BILATERAL": [\n      "M25.561",\n      "M25.562"\n    ],\n    "PRESENT_UNSPECIFIED": "M25.569"\n  },\n  "migraine": {\n    "PRESENT_UNSPECIFIED": "G43.909"\n  },\n  "mild nonproliferative diabetic retinopathy": {\n    "PRESENT_RIGHT": "E11.3291",\n    "PRESENT_LEFT": "E11.3292",\n    "PRESENT_BILATERAL": "E11.3293",\n    "PRESENT_UNSPECIFIED": "E11.3299"\n  },\n  "myocardial infarction": {\n    "PRESENT_UNSPECIFIED": "I21.9",\n    "HISTORICAL_UNSPECIFIED": "I25.2"\n  },\n  "nicotine dependence": {\n    "PRESENT_UNSPECIFIED": "F17.210",\n    "HISTORICAL_UNSPECIFIED": "Z87.891"\n  },\n  "obesity": {\n    "PRESENT_UNSPECIFIED": "E66.9"\n  },\n  "pneumonia": {\n    "PRESENT_UNSPECIFIED": "J18.9"\n  },\n  "shortness of breath": {\n    "PRESENT_UNSPECIFIED": "R06.02"\n  },\n  "shoulder pain": {\n    "PRESENT_RIGHT": "M25.511",\n    "PRESENT_LEFT": "M25.512",\n    "PRESENT_BILATERAL": [\n      "M25.511",\n      "M25.512"\n    ],\n    "PRESENT_UNSPECIFIED": "M25.519"\n  },\n  "stroke": {\n    "PRESENT_UNSPECIFIED": "I63.9",\n    "HISTORICAL_UNSPECIFIED": "Z86.73"\n  },\n  "type 1 diabetes": {\n    "PRESENT_UNSPECIFIED": "E10.9"\n  },\n  "type 2 diabetes": {\n    "PRESENT_UNSPECIFIED": "E11.9",\n    "FAMILY_HISTORY_UNSPECIFIED": "Z83.3"\n  },\n  "urinary tract infection": {\n    "PRESENT_UNSPECIFIED": "N39.0"\n  }\n}',
    'yappu_fsm/data/icd10cm_rules.ace': '# ICD-10-CM coding rules in the yappu coding-rule language (ACE-style controlled English).\n# One rule per line, optional "ID:" prefix. Concepts are lexicon entries written with hyphens.\n# Demo rule set: verify against the current ICD-10-CM Tabular List and Official Guidelines.\n\n# --- "With" convention (Guideline I.A.15): diabetes + CKD -> combination code, plus CKD stage\nDM_CKD: If the patient has a type-2-diabetes and the patient has a chronic-kidney-disease then the code E11.22 is assigned and the code E11.22 precedes the code N18.9.\n\n# --- Hypertension + CKD are presumed linked (Guideline I.C.9.a.2): I12.9 replaces I10\nHTN_CKD: If the patient has a hypertension and the patient has a chronic-kidney-disease then the code I12.9 is assigned and the code I10 is not assigned and the code I12.9 precedes the code N18.9.\n\n# --- E11.9 means "without complications": it cannot stand next to a complication code\nE11_9_A: If the code E11.22 is assigned then the code E11.9 is not assigned.\nE11_9_B: If the code E11.21 is assigned then the code E11.9 is not assigned.\nE11_9_C: If the code E11.319 is assigned then the code E11.9 is not assigned.\nE11_9_D: If the code E11.3291 is assigned then the code E11.9 is not assigned.\nE11_9_E: If the code E11.3292 is assigned then the code E11.9 is not assigned.\nE11_9_F: If the code E11.3293 is assigned then the code E11.9 is not assigned.\nE11_9_G: If the code E11.3299 is assigned then the code E11.9 is not assigned.\n\n# --- Excludes1 at E11 (type 1 diabetes, E10.-): report for coder review instead of guessing\nEXCL_E10_E11: If the code E10.9 is assigned and the code E11.9 is assigned then the codes E10.9 and E11.9 conflict.\n\n# --- Signs and symptoms integral to a confirmed diagnosis are not coded (Guideline I.B.5)\nSYMPT_PNA_COUGH: If the code J18.9 is assigned then the code R05.9 is not assigned.\nSYMPT_PNA_FEVER: If the code J18.9 is assigned then the code R50.9 is not assigned.\n\n# --- Inpatient uncertain diagnoses are coded as if established (Guideline II.H / III.C).\n#     Outpatient (Guideline IV.H) needs no rule: uncertain diagnoses simply get no code.\nINPT_UNCERTAIN: If the encounter is an inpatient-encounter and the patient has an uncertain diagnosis X then the candidate code of X is assigned.\n\n# --- Default sequencing: definitive diagnoses before symptom codes (R), symptoms before\n#     factor/history codes (Z). Guideline I.B.5 / Section IV.D: signs and symptoms are listed\n#     after a related definitive diagnosis; edit these lines to change the house policy.\nSEQ_DX_SYMPTOM: If a code C is assigned and the code C is a symptom code and a code D is assigned and the code D is a diagnosis code then the code D precedes the code C.\nSEQ_DX_FACTOR: If a code C is assigned and the code C is a factor code and a code D is assigned and the code D is a diagnosis code then the code D precedes the code C.\nSEQ_SYMPTOM_FACTOR: If a code C is assigned and the code C is a factor code and a code D is assigned and the code D is a symptom code then the code D precedes the code C.\n',
    'yappu_fsm/decoding.py': '"""Constrained decoding for Hugging Face ``model.generate()``.\n\nTwo logits processors are provided:\n\n* :class:`VenbaLogitsProcessor` – forces the generated continuation to be a\n  complete Nerisai Venba built from a :class:`~yappu_fsm.lexicon.VenbaLexicon`\n  (Vendalai linking, no Kanicheer, Ethukai rhyme, valid Eetrucheer) and adds a\n  soft logit boost for Monai alliteration.\n* :class:`ICD10LogitsProcessor` – after a marker such as ``"CODES:"`` the model\n  can only emit codes from an allowed set (e.g. the codes produced by the\n  clinical linker), separated by ``";"`` and ended by a newline or EOS.\n\nWhy word-level?  LLM tokens are subword pieces (often partial UTF-8 bytes for\nTamil), so the Asai of a single token says nothing about the word being built.\nBoth processors therefore work on the **bytes** of every vocabulary token and\nwalk a byte trie of whole allowed words / codes.  A token is allowed only if,\nappended to the text generated so far, it stays on a path that can still be\ncompleted.  Allowed-token sets are cached per decoder state.\n\nFormal property (checked by the tests): every sequence accepted by the decoder\nis a member of the allowed language – for Venba, ``validate_nerisai_venba``\nreturns VALID; for ICD-10, every emitted code is in the allowed set.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field, replace\nfrom typing import Dict, Iterable, List, Optional, Sequence, Tuple\n\nfrom .lexicon import Entry, VenbaLexicon\nfrom .prosody import LINE_STARTS\nfrom .tamil import monai_key\n\nSPACE_BYTES = {0x20, 0x0A}\n\n\n# ==============================================================================\n# Token -> raw bytes\n# ==============================================================================\n\n\ndef _bytes_to_unicode() -> Dict[int, str]:\n    """GPT-2 byte <-> printable unicode table (used by byte-level BPE)."""\n    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))\n    cs = bs[:]\n    n = 0\n    for b in range(256):\n        if b not in bs:\n            bs.append(b)\n            cs.append(256 + n)\n            n += 1\n    return dict(zip(bs, map(chr, cs)))\n\n\ndef _decoder_types(tokenizer) -> set:\n    try:\n        spec = json.loads(tokenizer.backend_tokenizer.to_str())\n    except Exception:\n        return set()\n    types = set()\n\n    def walk(node):\n        if isinstance(node, dict):\n            if "type" in node:\n                types.add(node["type"])\n            for v in node.values():\n                walk(v)\n        elif isinstance(node, list):\n            for v in node:\n                walk(v)\n\n    walk(spec.get("decoder"))\n    walk(spec.get("pre_tokenizer"))\n    return types\n\n\ndef token_bytes_table(tokenizer) -> List[Optional[bytes]]:\n    """Raw bytes produced by each token id (``None`` for special tokens)."""\n    vocab_size = len(tokenizer)\n    special = set(getattr(tokenizer, "all_special_ids", []) or [])\n    types = _decoder_types(tokenizer)\n    table: List[Optional[bytes]] = [None] * vocab_size\n    if "ByteLevel" in types:\n        inv = {c: b for b, c in _bytes_to_unicode().items()}\n        for tid in range(vocab_size):\n            if tid in special:\n                continue\n            tok = tokenizer.convert_ids_to_tokens(tid)\n            if tok is None:\n                continue\n            try:\n                table[tid] = bytes(inv[c] for c in tok)\n            except KeyError:\n                table[tid] = tok.encode("utf-8")\n        return table\n    for tid in range(vocab_size):\n        if tid in special:\n            continue\n        tok = tokenizer.convert_ids_to_tokens(tid)\n        if tok is None:\n            continue\n        if len(tok) == 6 and tok.startswith("<0x") and tok.endswith(">"):\n            table[tid] = bytes([int(tok[3:5], 16)])  # SentencePiece byte fallback\n        elif "Metaspace" in types or "▁" in tok:\n            table[tid] = tok.replace("▁", " ").encode("utf-8")\n        else:\n            table[tid] = tokenizer.decode([tid]).encode("utf-8")\n    return table\n\n\n# ==============================================================================\n# Byte trie\n# ==============================================================================\n\n\nclass TrieNode:\n    __slots__ = ("children", "value", "monai", "nid")\n\n    def __init__(self, nid: int):\n        self.children: Dict[int, "TrieNode"] = {}\n        self.value = None          # Entry (Venba) or str (ICD code) at a terminal node\n        self.monai: set = set()    # Monai keys of words in this subtree\n        self.nid = nid\n\n\nclass ByteTrie:\n    def __init__(self, items: Iterable[Tuple[str, object]]):\n        self._n = 0\n        self.root = self._new()\n        for text, value in items:\n            node = self.root\n            mk = monai_key(text)\n            node.monai.add(mk)\n            for b in text.encode("utf-8"):\n                node = node.children.get(b) or node.children.setdefault(b, self._new())\n                node.monai.add(mk)\n            node.value = value\n\n    def _new(self) -> TrieNode:\n        self._n += 1\n        return TrieNode(self._n)\n\n\n# ==============================================================================\n# Venba decoder state machine\n# ==============================================================================\n\n\n@dataclass(frozen=True)\nclass VenbaState:\n    k: int = 0                         # index of the foot being written (0..14)\n    expected: Optional[str] = None     # required first asai of this foot\n    key_a: Optional[str] = None        # Ethukai anchor of line 1\n    key_b: Optional[str] = None        # Ethukai anchor of line 3\n    line_monai: Optional[str] = None   # Monai key of the current line\'s first word\n    node_path: bytes = b""             # bytes of the current (partial) word\n    lead_space_ok: bool = True         # one leading space allowed before foot 0\n    words: Tuple[str, ...] = field(default_factory=tuple, compare=False)\n\n\nclass VenbaDecoder:\n    """Pure-Python state machine; usable without torch."""\n\n    def __init__(self, lexicon: VenbaLexicon, monai_boost: float = 3.0, pozhippu_factor: float = 1.5):\n        self.lex = lexicon\n        self.monai_boost = monai_boost\n        self.pozhippu_factor = pozhippu_factor\n        self._tries: Dict[tuple, ByteTrie] = {}\n\n    def trie_for(self, s: VenbaState) -> ByteTrie:\n        sig = (s.k, s.expected, s.key_a, s.key_b)\n        t = self._tries.get(sig)\n        if t is None:\n            t = ByteTrie((e.word, e) for e in self.lex.candidates(*sig))\n            self._tries[sig] = t\n        return t\n\n    def node(self, s: VenbaState) -> Optional[TrieNode]:\n        n = self.trie_for(s).root\n        for b in s.node_path:\n            n = n.children.get(b)\n            if n is None:\n                return None\n        return n\n\n    def initial(self) -> VenbaState:\n        return VenbaState()\n\n    def advance(self, s: VenbaState, data: bytes) -> Optional[VenbaState]:\n        node = self.node(s)\n        if node is None:\n            return None\n        for b in data:\n            if b in SPACE_BYTES:\n                if not s.node_path:\n                    if s.k == 0 and s.lead_space_ok and not s.words:\n                        s = replace(s, lead_space_ok=False)\n                        continue\n                    return None\n                e: Entry = node.value\n                if e is None or s.k == 14:\n                    return None\n                k = s.k\n                key_a = e.ethukai if k == 0 else s.key_a\n                key_b = e.ethukai if k == 8 else s.key_b\n                nk = k + 1\n                line_monai = e.monai if k in LINE_STARTS else s.line_monai\n                s = VenbaState(nk, self.lex._next_keys(k, e, s.key_a, s.key_b)[0], key_a, key_b,\n                               line_monai, b"", False, s.words + (e.word,))\n                node = self.trie_for(s).root\n                if not node.children:\n                    return None\n            else:\n                node = node.children.get(b)\n                if node is None:\n                    return None\n                s = replace(s, node_path=s.node_path + bytes([b]), lead_space_ok=False)\n        return s\n\n    def is_complete(self, s: VenbaState) -> bool:\n        if s.k != 14 or not s.node_path:\n            return False\n        n = self.node(s)\n        return n is not None and n.value is not None\n\n    def finished_words(self, s: VenbaState) -> List[str]:\n        out = list(s.words)\n        if self.is_complete(s):\n            out.append(self.node(s).value.word)\n        return out\n\n    def monai_target(self, s: VenbaState) -> Optional[str]:\n        return None if s.k in LINE_STARTS else s.line_monai\n\n\nclass VenbaLogitsProcessor:\n    """Hugging Face logits processor wrapping :class:`VenbaDecoder`.\n\n    Call :meth:`reset` (or build a new processor) before each ``generate``.\n    """\n\n    def __init__(self, tokenizer, decoder: VenbaDecoder, eos_token_id: Optional[int] = None):\n        import torch  # noqa: F401  (fail early with a clear message if torch is missing)\n\n        self.decoder = decoder\n        self.token_bytes = token_bytes_table(tokenizer)\n        self.eos = tokenizer.eos_token_id if eos_token_id is None else eos_token_id\n        self._cache: Dict[VenbaState, Tuple[list, list, list]] = {}\n        self.prompt_len: Optional[int] = None\n\n    def reset(self):\n        self.prompt_len = None\n\n    def state_for(self, generated: Sequence[int]) -> Optional[VenbaState]:\n        s = self.decoder.initial()\n        for tid in generated:\n            if tid == self.eos:\n                break\n            data = self.token_bytes[tid] if tid < len(self.token_bytes) else None\n            if data is None:\n                return None\n            s = self.decoder.advance(s, data)\n            if s is None:\n                return None\n        return s\n\n    def allowed(self, s: VenbaState):\n        hit = self._cache.get(s)\n        if hit is not None:\n            return hit\n        allowed, boosted, pozhippu = [], [], []\n        for tid, data in enumerate(self.token_bytes):\n            if not data:\n                continue\n            ns = self.decoder.advance(s, data)\n            if ns is None:\n                continue\n            allowed.append(tid)\n            if ns.node_path and (ns.k > s.k or not s.node_path):\n                target = self.decoder.monai_target(ns)\n                n = self.decoder.node(ns)\n                if target and n is not None and target in n.monai:\n                    is_pozhippu = ns.k - self.decoder.lex.line_start(ns.k) == 2\n                    (pozhippu if is_pozhippu else boosted).append(tid)\n        if self.decoder.is_complete(s) and self.eos is not None:\n            allowed.append(self.eos)\n        self._cache[s] = (allowed, boosted, pozhippu)\n        return self._cache[s]\n\n    def __call__(self, input_ids, scores):\n        import torch\n\n        if self.prompt_len is None:\n            self.prompt_len = input_ids.shape[1]\n        for b in range(input_ids.shape[0]):\n            gen = input_ids[b, self.prompt_len:].tolist()\n            s = self.state_for(gen)\n            mask = torch.full_like(scores[b], float("-inf"))\n            if s is None:\n                if self.eos is not None:\n                    mask[self.eos] = 0.0\n            else:\n                allowed, boosted, pozhippu = self.allowed(s)\n                if not allowed and self.eos is not None:\n                    allowed = [self.eos]\n                mask[allowed] = 0.0\n                if boosted:\n                    mask[boosted] += self.decoder.monai_boost\n                if pozhippu:\n                    mask[pozhippu] += self.decoder.monai_boost * self.decoder.pozhippu_factor\n            scores[b] = scores[b] + mask\n        return scores\n\n\n# ==============================================================================\n# ICD-10 constrained code emission\n# ==============================================================================\n\n\nclass ICD10CodeTrie:\n    def __init__(self, codes: Iterable[str]):\n        self.codes = sorted({c.strip().upper() for c in codes if c.strip()})\n        self.trie = ByteTrie((c, c) for c in self.codes)\n\n    def walk(self, prefix: str) -> Optional[TrieNode]:\n        n = self.trie.root\n        for b in prefix.encode("utf-8"):\n            n = n.children.get(b)\n            if n is None:\n                return None\n        return n\n\n    def is_code(self, s: str) -> bool:\n        n = self.walk(s)\n        return n is not None and n.value is not None\n\n\nclass ICD10LogitsProcessor:\n    """After ``marker`` appears in the generated text (or at the end of the\n    prompt), restrict output to ``CODE[; CODE]*`` then newline / EOS, where every\n    CODE is in ``allowed_codes``.  Duplicate codes are not allowed."""\n\n    def __init__(self, tokenizer, allowed_codes: Iterable[str], marker: str = "CODES:", separator: str = ";"):\n        import torch  # noqa: F401\n\n        self.tok = tokenizer\n        self.trie = ICD10CodeTrie(allowed_codes)\n        self.marker = marker\n        self.sep = separator\n        self.eos = tokenizer.eos_token_id\n        self.token_bytes = token_bytes_table(tokenizer)\n        self.token_text = [b.decode("utf-8", errors="replace") if b else None for b in self.token_bytes]\n        self._cache: Dict[Tuple[str, frozenset], List[int]] = {}\n\n    # state = (partial code, codes already emitted); returns None if invalid\n    def _advance(self, partial: str, done: frozenset, text: str):\n        for ch in text:\n            if ch == " ":\n                if partial:\n                    return None\n                continue\n            if ch in (self.sep, "\\n"):\n                if not self.trie.is_code(partial) or partial in done:\n                    return None\n                done, partial = done | {partial}, ""\n                if ch == "\\n":\n                    return ("<END>", done)\n                if not self._live("", done):        # no unused code left to follow the separator\n                    return None\n                continue\n            partial += ch\n            if not self._live(partial, done):\n                return None\n        return (partial, done)\n\n    def _live(self, partial: str, done: frozenset) -> bool:\n        """Can ``partial`` still be completed to a code that has not been emitted yet?"""\n        return any(c.startswith(partial) and c not in done for c in self.trie.codes)\n\n    def allowed(self, partial: str, done: frozenset) -> List[int]:\n        key = (partial, done)\n        if key in self._cache:\n            return self._cache[key]\n        out = [tid for tid, text in enumerate(self.token_text)\n               if text and self._advance(partial, done, text) is not None]\n        if self.trie.is_code(partial) and partial not in done and self.eos is not None:\n            out.append(self.eos)\n        self._cache[key] = out\n        return out\n\n    def parse_state(self, text: str):\n        idx = text.rfind(self.marker)\n        if idx < 0:\n            return None\n        body = text[idx + len(self.marker):]\n        if "\\n" in body:\n            return "<END>", frozenset()\n        parts = body.split(self.sep)\n        done = frozenset(p.strip() for p in parts[:-1])\n        return parts[-1].strip(), done\n\n    def __call__(self, input_ids, scores):\n        import torch\n\n        for b in range(input_ids.shape[0]):\n            text = self.tok.decode(input_ids[b], skip_special_tokens=True)\n            st = self.parse_state(text)\n            if st is None:\n                continue\n            partial, done = st\n            mask = torch.full_like(scores[b], float("-inf"))\n            if partial == "<END>":\n                if self.eos is not None:\n                    mask[self.eos] = 0.0\n            else:\n                ids = self.allowed(partial, done) or ([self.eos] if self.eos is not None else [])\n                mask[ids] = 0.0\n            scores[b] = scores[b] + mask\n        return scores\n\n\ndef parse_codes(text: str, marker: str = "CODES:", separator: str = ";") -> List[str]:\n    idx = text.rfind(marker)\n    if idx < 0:\n        return []\n    body = text[idx + len(marker):].split("\\n")[0]\n    return [c.strip() for c in body.split(separator) if c.strip()]\n',
    'yappu_fsm/demo.py': '"""Offline constrained-decoding demo.\n\nThe Hugging Face hub is not needed: a byte-level BPE tokenizer is trained on the\nlexicon and attached to a *randomly initialised* tiny GPT-2.  The model knows\nnothing, yet every generation is a valid Nerisai Venba / uses only allowed\nICD-10 codes – the guarantee comes from the logits processors, not the model.\nSwap in any real causal LM (``AutoModelForCausalLM.from_pretrained(...)``) and\nits tokenizer to get fluent output under the same guarantees.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Iterable, Tuple\n\n\ndef build_tiny_model(corpus: Iterable[str], seed: int = 0, vocab_size: int = 400) -> Tuple[object, object]:\n    import torch\n    from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers\n    from transformers import GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast\n\n    tok = Tokenizer(models.BPE())\n    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)\n    tok.decoder = decoders.ByteLevel()\n    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["<eos>"],\n                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())\n    tok.train_from_iterator(list(corpus), trainer)\n    hf_tok = PreTrainedTokenizerFast(tokenizer_object=tok, eos_token="<eos>", pad_token="<eos>")\n    torch.manual_seed(seed)\n    cfg = GPT2Config(vocab_size=len(hf_tok), n_positions=512, n_embd=32, n_layer=1, n_head=2,\n                     eos_token_id=hf_tok.eos_token_id, bos_token_id=hf_tok.eos_token_id,\n                     pad_token_id=hf_tok.eos_token_id)\n    model = GPT2LMHeadModel(cfg).eval()\n    return hf_tok, model\n\n\ndef generate_venba(tokenizer, model, lexicon=None, prompt: str = "வெண்பா:", do_sample: bool = True,\n                   seed: int = 0, max_new_tokens: int = 400):\n    import torch\n    from transformers import LogitsProcessorList\n\n    from .decoding import VenbaDecoder, VenbaLogitsProcessor\n    from .lexicon import VenbaLexicon\n\n    lexicon = lexicon or VenbaLexicon.sample()\n    proc = VenbaLogitsProcessor(tokenizer, VenbaDecoder(lexicon))\n    ids = tokenizer(prompt, return_tensors="pt")\n    torch.manual_seed(seed)\n    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=do_sample, top_k=0,\n                         logits_processor=LogitsProcessorList([proc]),\n                         eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)\n    gen = out[0, ids["input_ids"].shape[1]:].tolist()\n    state = proc.state_for(gen)\n    words = proc.decoder.finished_words(state) if state is not None else []\n    return words, tokenizer.decode(gen, skip_special_tokens=True)\n\n\ndef generate_codes(tokenizer, model, allowed_codes, prompt: str, seed: int = 0, max_new_tokens: int = 80):\n    import torch\n    from transformers import LogitsProcessorList\n\n    from .decoding import ICD10LogitsProcessor, parse_codes\n\n    proc = ICD10LogitsProcessor(tokenizer, allowed_codes)\n    ids = tokenizer(prompt, return_tensors="pt")\n    torch.manual_seed(seed)\n    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True, top_k=0,\n                         logits_processor=LogitsProcessorList([proc]),\n                         eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)\n    text = tokenizer.decode(out[0], skip_special_tokens=True)\n    return parse_codes(text), text\n\n\ndef run_demo(seed: int = 0):\n    from . import data_path\n    from .clinical import ClinicalLinker, load_lexicon\n    from .lexicon import SAMPLE_WORDS\n    from .prosody import format_venba, validate_nerisai_venba\n\n    lex = load_lexicon(data_path("icd10cm_example_lexicon.json"))\n    codes_corpus = [c for v in lex.values() for vv in v.values() for c in ([vv] if isinstance(vv, str) else vv)]\n    corpus = [" ".join(SAMPLE_WORDS)] * 4 + ["CODES: " + "; ".join(codes_corpus)] * 4\n    tok, model = build_tiny_model(corpus, seed=seed)\n\n    print("== Constrained Nerisai Venba from a random tiny GPT-2 ==")\n    words, _ = generate_venba(tok, model, seed=seed)\n    print(format_venba(words))\n    rep = validate_nerisai_venba(words)\n    print("validator:", "VALID" if rep.valid else "INVALID", rep.vikarpam or "")\n\n    print("\\n== Constrained ICD-10 emission ==")\n    linker = ClinicalLinker(lex)\n    note = "Pt has lt knee osteoarthritis. Denies SOB, but has uncontrolled HTN. R/O CAP."\n    allowed = linker.allowed_codes(linker.process(note))\n    codes, text = generate_codes(tok, model, allowed, prompt=f"Note: {note}\\nCODES:", seed=seed)\n    print("note:          ", note)\n    print("allowed codes: ", allowed)\n    print("model emitted: ", codes)\n',
    'yappu_fsm/lexicon.py': '"""Annotated word lexicon used by the constrained Venba decoder."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom functools import lru_cache\nfrom typing import Dict, Iterable, List, Optional, Tuple\n\nfrom .prosody import EETRUCHEER, LINE_STARTS, VENBA_CATEGORIES, classify, eetrucheer_vaipadu, expected_next_start\nfrom .tamil import encode_asai, ethukai_key, monai_key, normalize\n\n\n@dataclass(frozen=True)\nclass Entry:\n    word: str\n    asai: str\n    category: Optional[str]\n    final_ok: bool          # usable as Eetrucheer\n    ethukai: Optional[str]\n    monai: Optional[str]\n\n\n# Words from the three poems in this project plus a handful of extras.\nSAMPLE_WORDS = """\nஎண்ணும் எழுத்தும் இசையோடு கூடினால் பண்ணும் மணியாகும் பார்ப்பவர் கண்ணும்\nஅமுதத் தமிழில் அமைந்த இலக்கணம் நம்மொழி யாக்கும் இசை\nவண்ண நிலவொளி வானில் தவழ்ந்திட எண்ணற்ற விண்மீன்கள் ஏற்றிடும் விண்ணில்\nஇருளினை நீக்கும் இனிய ஒளியால் மருளினை மாற்றும் மதி\nஅடக்க முடையா ரறிவிலரென் றெண்ணிக் கடக்கக் கருதவும் வேண்டா மடைத்தலையில்\nஓடுமீ னோட உறுமீன் வருமளவும் வாடி யிருக்குமாம் கொக்கு\nமண்ணில் கண்ணில் பொருளினை அருளினை மலர் நாள் காசு பிறப்பு அன்பு\n""".split()\n\n\nclass VenbaLexicon:\n    """A set of words annotated with Asai, Seer category, Ethukai and Monai keys."""\n\n    def __init__(self, words: Iterable[str], strict_ethukai: bool = False, scheme: str = "iru"):\n        if scheme not in ("iru", "oru"):\n            raise ValueError("scheme must be \'iru\' or \'oru\'")\n        self.strict_ethukai = strict_ethukai\n        self.scheme = scheme\n        seen, entries = set(), []\n        for w in words:\n            w = normalize(w.strip())\n            if not w or w in seen:\n                continue\n            seen.add(w)\n            asai = encode_asai(w)\n            if not asai:\n                continue\n            entries.append(Entry(w, asai, classify(asai)[1], eetrucheer_vaipadu(w) is not None,\n                                 ethukai_key(w, strict_ethukai), monai_key(w)))\n        self.entries: List[Entry] = entries\n        self.by_word: Dict[str, Entry] = {e.word: e for e in entries}\n        self._viable = lru_cache(maxsize=None)(self._viable_uncached)\n\n    @classmethod\n    def sample(cls, **kw) -> "VenbaLexicon":\n        return cls(SAMPLE_WORDS, **kw)\n\n    # ---- constraint logic -------------------------------------------------\n\n    def _passes(self, e: Entry, k: int, expected: Optional[str], key_a: Optional[str], key_b: Optional[str]) -> bool:\n        if k == EETRUCHEER:\n            if not e.final_ok:\n                return False\n            # Kaasu/Pirappu start like Naal/Malar\n        elif e.category not in VENBA_CATEGORIES:\n            return False\n        if expected is not None and not e.asai.startswith(expected):\n            return False\n        if k in (4, 7) and e.ethukai != key_a:\n            return False\n        if k == 8 and self.scheme == "oru" and e.ethukai != key_a:\n            return False\n        if k == 12 and e.ethukai != key_b:\n            return False\n        if k in (0, 8) and e.ethukai is None:\n            return False\n        return True\n\n    def _next_keys(self, k: int, e: Entry, key_a, key_b):\n        if k == 0:\n            key_a = e.ethukai\n        if k == 8:\n            key_b = e.ethukai\n        return expected_next_start(e.asai), key_a, key_b\n\n    def _viable_uncached(self, k: int, expected, key_a, key_b) -> bool:\n        """Is there at least one way to finish the poem from foot ``k``?"""\n        for e in self.entries:\n            if self._passes(e, k, expected, key_a, key_b):\n                if k == EETRUCHEER:\n                    return True\n                if self._viable(k + 1, *self._next_keys(k, e, key_a, key_b)):\n                    return True\n        return False\n\n    @lru_cache(maxsize=None)\n    def candidates(self, k: int, expected, key_a, key_b) -> Tuple[Entry, ...]:\n        """Words allowed at foot ``k`` that still leave a complete poem reachable."""\n        out = []\n        for e in self.entries:\n            if not self._passes(e, k, expected, key_a, key_b):\n                continue\n            if k == EETRUCHEER or self._viable(k + 1, *self._next_keys(k, e, key_a, key_b)):\n                out.append(e)\n        return tuple(out)\n\n    def line_start(self, k: int) -> int:\n        return max(s for s in LINE_STARTS if s <= k)\n',
    'yappu_fsm/prosody.py': '"""Seer classification, Thalai (linking) rules and a full Nerisai Venba validator."""\n\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nfrom .tamil import encode_asai, ethukai_key, has_kutriyalukaram_ending, split_display\n\n# asai string -> (Vaipadu, category)\nSEER: Dict[str, Tuple[str, str]] = {\n    "11": ("Thema", "Maacheer"),\n    "01": ("Pulima", "Maacheer"),\n    "00": ("Karuvilam", "Vilacheer"),\n    "10": ("Koovilam", "Vilacheer"),\n    "111": ("Themangai", "Kaycheer"),\n    "011": ("Pulimangai", "Kaycheer"),\n    "001": ("Karuvilangai", "Kaycheer"),\n    "101": ("Koovilangai", "Kaycheer"),\n    "110": ("Themangani", "Kanicheer"),\n    "010": ("Pulimangani", "Kanicheer"),\n    "000": ("Karuvilangani", "Kanicheer"),\n    "100": ("Koovilangani", "Kanicheer"),\n}\nVENBA_CATEGORIES = {"Maacheer", "Vilacheer", "Kaycheer"}\n\n# Nerisai Venba layout: foot indices per line; foot 7 is the Thanichol, foot 14 the Eetrucheer.\nLINE_FEET = [(0, 1, 2, 3), (4, 5, 6, 7), (8, 9, 10, 11), (12, 13, 14)]\nTHANICHOL, EETRUCHEER = 7, 14\nLINE_STARTS = (0, 4, 8, 12)\n\n\ndef classify(asai: str) -> Tuple[Optional[str], Optional[str]]:\n    return SEER.get(asai, (None, None))\n\n\ndef eetrucheer_vaipadu(word: str) -> Optional[str]:\n    """Return Naal / Malar / Kaasu / Pirappu if ``word`` is a valid final foot."""\n    asai = encode_asai(word)\n    if asai == "1":\n        return "Naal"\n    if asai == "0":\n        return "Malar"\n    if has_kutriyalukaram_ending(word):\n        if asai == "11":\n            return "Kaasu"\n        if asai == "01":\n            return "Pirappu"\n    return None\n\n\ndef expected_next_start(prev_asai: str) -> Optional[str]:\n    """Vendalai: after Maacheer the next foot starts with Nirai (\'0\'); after\n    Vilacheer or Kaycheer it starts with Ner (\'1\')."""\n    cat = classify(prev_asai)[1]\n    if cat == "Maacheer":\n        return "0"\n    if cat in ("Vilacheer", "Kaycheer"):\n        return "1"\n    return None\n\n\ndef thalai_name(prev_asai: str, next_asai: str) -> str:\n    cat = classify(prev_asai)[1]\n    start = next_asai[:1]\n    if cat == "Maacheer":\n        return "Iyarseer Vendalai" if start == "0" else "Nerondru Aasiriyathalai"\n    if cat == "Vilacheer":\n        return "Iyarseer Vendalai" if start == "1" else "Niraiondru Aasiriyathalai"\n    if cat == "Kaycheer":\n        return "Venseer Vendalai" if start == "1" else "Kalithalai"\n    if cat == "Kanicheer":\n        return "Ondriya Vanjithalai" if start == "0" else "Ondraadha Vanjithalai"\n    return "Unknown"\n\n\ndef is_vendalai(name: str) -> bool:\n    return name in ("Iyarseer Vendalai", "Venseer Vendalai")\n\n\n@dataclass\nclass FootReport:\n    index: int\n    word: str\n    asai: str\n    split: str\n    vaipadu: Optional[str]\n    category: Optional[str]\n    thalai_in: Optional[str]\n    ok: bool\n    message: str = ""\n\n\n@dataclass\nclass VenbaReport:\n    valid: bool\n    feet: List[FootReport] = field(default_factory=list)\n    errors: List[str] = field(default_factory=list)\n    vikarpam: Optional[str] = None\n\n    def lines(self) -> List[str]:\n        out = []\n        for f in self.feet:\n            mark = "ok " if f.ok else "ERR"\n            out.append(\n                f"[{mark}] foot {f.index:02d} {f.word:<14} {f.split:<16} {f.asai:<4} "\n                f"{(f.vaipadu or \'-\'):<13} {(f.thalai_in or \'-\'):<26} {f.message}"\n            )\n        out += [f"ERROR: {e}" for e in self.errors]\n        out.append(f"Verdict: {\'VALID\' if self.valid else \'INVALID\'} Nerisai Venba"\n                   + (f" ({self.vikarpam})" if self.vikarpam else ""))\n        return out\n\n\ndef parse_venba(text: str) -> List[str]:\n    """Split poem text into words, dropping punctuation such as the Thanichol dash."""\n    return [w for w in re.split(r"[\\s\\-–—.,;!?]+", text) if w]\n\n\ndef format_venba(words: Sequence[str]) -> str:\n    """Lay 15 feet out as a 4-line Nerisai Venba with the Thanichol dash."""\n    w = list(words)\n    if len(w) != 15:\n        return " ".join(w)\n    return "\\n".join([\n        " ".join(w[0:4]),\n        " ".join(w[4:7]) + " - " + w[7],\n        " ".join(w[8:12]),\n        " ".join(w[12:15]) + ".",\n    ])\n\n\ndef validate_nerisai_venba(poem, strict_ethukai: bool = False) -> VenbaReport:\n    """Validate a Nerisai Venba given as text or as a list of 15 words.\n\n    Checks: 15 feet; only Maa/Vilam/Kay feet (no Kanicheer, no 4-asai feet);\n    a valid Orasai/Kaasu/Pirappu Eetrucheer; Vendalai on all 14 links; and\n    Ethukai (Oru or Iru Vikarpam) across lines 1-2 + Thanichol and lines 3-4.\n    """\n    words = parse_venba(poem) if isinstance(poem, str) else list(poem)\n    rep = VenbaReport(valid=True)\n    if len(words) != 15:\n        rep.valid = False\n        rep.errors.append(f"Expected 15 feet (4+4+4+3), got {len(words)}.")\n        return rep\n\n    prev = None\n    for i, w in enumerate(words):\n        asai = encode_asai(w)\n        ok, msg = True, ""\n        if i == EETRUCHEER:\n            vp = eetrucheer_vaipadu(w)\n            cat = "Eetrucheer" if vp else None\n            if not vp:\n                ok, msg = False, "final foot must be Naal/Malar/Kaasu/Pirappu"\n        else:\n            vp, cat = classify(asai)\n            if cat not in VENBA_CATEGORIES:\n                ok = False\n                msg = f"{vp or asai} ({cat or \'unclassified\'}) not allowed in Venba"\n        th = None\n        if prev is not None:\n            th = thalai_name(prev, asai)\n            if not is_vendalai(th):\n                ok = False\n                msg = (msg + "; " if msg else "") + f"link is {th}"\n        rep.feet.append(FootReport(i, w, asai, split_display(w), vp, cat, th, ok, msg))\n        rep.valid &= ok\n        prev = asai\n\n    a = [ethukai_key(words[i], strict_ethukai) for i in (0, 4, THANICHOL)]\n    b = [ethukai_key(words[i], strict_ethukai) for i in (8, 12)]\n    if None in a or len(set(a)) != 1:\n        rep.valid = False\n        rep.errors.append(f"Ethukai A mismatch across feet 0, 4, 7: {a}")\n    if None in b or len(set(b)) != 1:\n        rep.valid = False\n        rep.errors.append(f"Ethukai B mismatch across feet 8, 12: {b}")\n    if rep.valid:\n        rep.vikarpam = "Oru Vikarpam" if a[0] == b[0] else "Iru Vikarpam"\n    return rep\n',
    'yappu_fsm/rules/__init__.py': '"""Controlled-English (ACE-style) coding rules compiled to Datalog."""\nfrom .ace import AceError, AceRule, compile_rules, parse_rule, parse_rules\nfrom .coder import CodingResult, RuleCoder\nfrom .datalog import DatalogError, Literal, Program, Rule, Var\n\n__all__ = ["AceError", "AceRule", "compile_rules", "parse_rule", "parse_rules", "CodingResult", "RuleCoder",\n           "DatalogError", "Literal", "Program", "Rule", "Var"]\n',
    'yappu_fsm/rules/ace.py': '"""A controlled-English rule language for ICD-10-CM coding, compiled to Datalog.\n\nThe sentences follow the conventions of Attempto Controlled English (ACE): a\nfixed set of function words, user-defined content words (here: lexicon concepts\nwritten with hyphens, e.g. ``type-2-diabetes``), simple present tense, a\ndeterministic reading shown back as a bracketed paraphrase, and a precise error\nwhen a sentence is outside the language.  It is a small *coding-rule subset*\nin the spirit of ACE, parsed by this module; it does not call the Attempto\nParsing Engine.\n\nGrammar (one rule per sentence)::\n\n    rule     := "If" cond ("and" cond)* "then" conseq ("and" conseq)* "."\n    cond     := subj "has" term\n              | subj "does not have" term\n              | subj "has" ("a"|"an") "uncertain" (CONCEPT | "diagnosis" VAR)\n              | subj "has a history of" term\n              | "a relative of the patient has" term\n              | "the encounter is" ("a"|"an") ("outpatient-encounter" | "inpatient-encounter")\n              | "the" (CONCEPT|VAR) "is on the" ("left"|"right") "side"\n              | "the" (CONCEPT|VAR) "is on both sides"\n              | "the code" (CODE|VAR) "is" ["not"] "assigned"\n              | "a code" VAR "is assigned"                      (introduces a code variable)\n              | "the code" (CODE|VAR) "is a" KIND "code"       KIND: diagnosis | symptom | factor | external\n    conseq   := "the code" CODE "is" ["not"] "assigned"\n              | "the candidate code of" VAR "is assigned"\n              | "the code" (CODE|VAR) "precedes the code" (CODE|VAR)\n              | "the codes" CODE "and" CODE "conflict"\n    subj     := ("a"|"the") "patient"\n    term     := ("a"|"an") CONCEPT | ("a"|"an") "diagnosis" VAR | VAR | "the" CONCEPT\n\nDatalog vocabulary produced: patient/1 (implicit in every rule), has/2, uncertain/2, history/2, family/2, setting/2,\nside/3, candidate/3, kind/3, code/2, blocked/2, precedes/3, conflict/3 and the built-in\n``final(P, C) :- code(P, C), not blocked(P, C).``\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass, field\nfrom typing import Iterable, List, Optional, Sequence, Set, Tuple\n\nfrom .datalog import Literal, Program, Rule, Var\n\nCODE_RE = re.compile(r"^[A-Z][0-9][0-9A-Z](?:\\.[0-9A-Z]{1,4})?$")\nVAR_RE = re.compile(r"^[A-Z][0-9]?$")\nCONCEPT_RE = re.compile(r"^[a-z0-9]+(?:-[a-z0-9]+)*$")\nP = Var("P")\nSETTINGS = {"outpatient-encounter": "outpatient", "inpatient-encounter": "inpatient"}\nSIDES = {"left": "left", "right": "right", "both": "both"}\nKINDS = ("diagnosis", "symptom", "factor", "external")\nFINAL_RULE = Rule(Literal("final", (P, Var("C"))), [Literal("code", (P, Var("C"))), Literal("blocked", (P, Var("C")), True)],\n                  "final", "A code is final if it is assigned and not blocked.")\n\n\nclass AceError(ValueError):\n    def __init__(self, sentence: str, pos: int, expected: str, found: str, line: Optional[int] = None):\n        self.sentence, self.pos, self.expected, self.found, self.line = sentence, pos, expected, found, line\n        where = f"line {line}: " if line else ""\n        super().__init__(f"{where}not in the coding-rule language: expected {expected} at word {pos + 1}, "\n                         f"found \'{found}\'\\n    {sentence}")\n\n\n@dataclass\nclass Atom:\n    kind: str                  # has | not_has | uncertain | history | family | setting | side | code | not_code\n    args: Tuple[object, ...]\n\n    def paraphrase(self) -> str:\n        a = [_say(x) for x in self.args] + ["", ""]\n        return {\n            "has": f"the patient has {a[0]}",\n            "not_has": f"it is false that the patient has {a[0]}",\n            "uncertain": f"the patient has an uncertain {a[0]}",\n            "history": f"the patient has a history of {a[0]}",\n            "family": f"a relative of the patient has {a[0]}",\n            "setting": f"the encounter is {a[0]}",\n            "side": f"{a[0]} is on side {a[1]}",\n            "code": f"code {a[0]} is assigned",\n            "kind": f"code {a[0]} is a {a[1]} code",\n            "not_code": f"it is false that code {a[0]} is assigned",\n            "assign": f"code {a[0]} is assigned",\n            "assign_candidate": f"the candidate code of {a[0]} is assigned",\n            "block": f"code {a[0]} is blocked",\n            "precedes": f"code {a[0]} is sequenced before code {a[1]}",\n            "conflict": f"codes {a[0]} and {a[1]} conflict",\n        }[self.kind]\n\n\ndef _say(x) -> str:\n    return x.name if isinstance(x, Var) else str(x).replace("-", " ")\n\n\n@dataclass\nclass AceRule:\n    rid: str\n    text: str\n    conds: List[Atom]\n    conseqs: List[Atom]\n\n    def paraphrase(self) -> str:\n        c = " AND ".join("{ " + a.paraphrase() + " }" for a in self.conds)\n        t = " AND ".join("{ " + a.paraphrase() + " }" for a in self.conseqs)\n        return f"IF {c} THEN {t}"\n\n    def to_datalog(self) -> List[Rule]:\n        body: List[Literal] = [Literal("patient", (P,))]   # binds P even when all conditions are negative\n        for a in self.conds:\n            k, args = a.kind, a.args\n            if k == "not_has":\n                body.append(Literal("has", (P,) + args, True))\n            elif k == "not_code":\n                body.append(Literal("code", (P,) + args, True))\n            else:\n                body.append(Literal(k, (P,) + args))\n        rules = []\n        for i, c in enumerate(self.conseqs):\n            rid = self.rid if len(self.conseqs) == 1 else f"{self.rid}.{i + 1}"\n            if c.kind == "assign":\n                rules.append(Rule(Literal("code", (P,) + c.args), list(body), rid, self.text))\n            elif c.kind == "assign_candidate":\n                cv = Var("C_" + c.args[0].name)\n                rules.append(Rule(Literal("code", (P, cv)), body + [Literal("candidate", (P, c.args[0], cv))], rid, self.text))\n            elif c.kind == "block":\n                rules.append(Rule(Literal("blocked", (P,) + c.args), list(body), rid, self.text))\n            elif c.kind in ("precedes", "conflict"):\n                rules.append(Rule(Literal(c.kind, (P,) + c.args), list(body), rid, self.text))\n        return rules\n\n\nclass _Parser:\n    def __init__(self, sentence: str, concepts: Optional[Set[str]], line: Optional[int]):\n        self.s = sentence.strip()\n        self.toks = re.findall(r"[A-Za-z0-9][A-Za-z0-9.\\-]*[A-Za-z0-9]|[A-Za-z0-9]|[.,]", self.s)\n        if self.toks and self.toks[-1] != "." and self.toks[-1].endswith("."):\n            self.toks[-1] = self.toks[-1][:-1]\n            self.toks.append(".")\n        self.i, self.concepts, self.line = 0, concepts, line\n        self.vars: Set[str] = set()\n\n    # helpers -------------------------------------------------------------\n    def peek(self, k=0) -> str:\n        j = self.i + k\n        return self.toks[j] if j < len(self.toks) else "<end>"\n\n    def fail(self, expected: str):\n        raise AceError(self.s, self.i, expected, self.peek(), self.line)\n\n    def word(self, *options: str) -> str:\n        t = self.peek()\n        if t.lower() in options:\n            self.i += 1\n            return t.lower()\n        self.fail(" or ".join(f"\'{o}\'" for o in options))\n\n    def phrase(self, text: str):\n        for w in text.split():\n            self.word(w)\n\n    def at(self, text: str) -> bool:\n        ws = text.split()\n        return all(self.peek(k).lower() == w for k, w in enumerate(ws))\n\n    def code(self) -> str:\n        t = self.peek()\n        if not CODE_RE.match(t):\n            self.fail("an ICD-10-CM code such as E11.22")\n        self.i += 1\n        return t\n\n    def code_or_var(self):\n        t = self.peek()\n        if CODE_RE.match(t):\n            return self.code()\n        if VAR_RE.match(t):\n            return self.var()\n        self.fail("an ICD-10-CM code or a code variable")\n\n    def concept(self) -> str:\n        t = self.peek()\n        if not CONCEPT_RE.match(t) or t in ("diagnosis", "uncertain", "history", "patient"):\n            self.fail("a concept word such as type-2-diabetes")\n        if self.concepts is not None and t.replace("-", " ") not in self.concepts:\n            raise AceError(self.s, self.i, "a concept from the lexicon", t, self.line)\n        self.i += 1\n        return t\n\n    def var(self, introduce=False) -> Var:\n        t = self.peek()\n        if not VAR_RE.match(t):\n            self.fail("a variable such as X")\n        if not introduce and t not in self.vars:\n            raise AceError(self.s, self.i, "a variable introduced earlier (\'a diagnosis X\')", t, self.line)\n        self.vars.add(t)\n        self.i += 1\n        return Var(t)\n\n    # grammar -------------------------------------------------------------\n    def term(self):\n        t = self.peek().lower()\n        if t in ("a", "an"):\n            self.i += 1\n            if self.peek().lower() == "diagnosis":\n                self.i += 1\n                return self.var(introduce=True)\n            return self.concept()\n        if t == "the":\n            self.i += 1\n            return self.concept()\n        if VAR_RE.match(self.peek()):\n            return self.var()\n        self.fail("\'a\', \'an\', \'the\' or a variable")\n\n    def cond(self) -> Atom:\n        t = self.peek().lower()\n        if t in ("a", "the") and self.peek(1).lower() == "patient":\n            self.i += 2\n            if self.at("does not have"):\n                self.phrase("does not have")\n                return Atom("not_has", (self.term(),))\n            self.word("has")\n            if self.at("a history of"):\n                self.phrase("a history of")\n                return Atom("history", (self.term(),))\n            if self.peek().lower() in ("a", "an") and self.peek(1).lower() == "uncertain":\n                self.i += 2\n                if self.peek().lower() == "diagnosis":\n                    self.i += 1\n                    return Atom("uncertain", (self.var(introduce=True),))\n                return Atom("uncertain", (self.concept(),))\n            return Atom("has", (self.term(),))\n        if self.at("a relative of the patient has"):\n            self.phrase("a relative of the patient has")\n            return Atom("family", (self.term(),))\n        if self.at("the encounter is"):\n            self.phrase("the encounter is")\n            self.word("a", "an")\n            s = self.word(*SETTINGS)\n            return Atom("setting", (SETTINGS[s],))\n        if self.at("a code"):\n            self.phrase("a code")\n            v = self.var(introduce=True)\n            self.phrase("is assigned")\n            return Atom("code", (v,))\n        if self.at("the code"):\n            self.phrase("the code")\n            c = self.code_or_var()\n            self.word("is")\n            if self.peek().lower() == "a":\n                self.i += 1\n                k = self.word(*KINDS)\n                self.word("code")\n                return Atom("kind", (c, k))\n            neg = self.peek().lower() == "not"\n            if neg:\n                self.i += 1\n            self.word("assigned")\n            return Atom("not_code" if neg else "code", (c,))\n        if t == "the":\n            self.i += 1\n            subj = self.var() if VAR_RE.match(self.peek()) else self.concept()\n            self.phrase("is on")\n            if self.peek().lower() == "both":\n                self.phrase("both sides")\n                return Atom("side", (subj, "both"))\n            self.word("the")\n            s = self.word("left", "right")\n            self.word("side")\n            return Atom("side", (subj, s))\n        self.fail("a condition (\'the patient has ...\', \'the code ... is assigned\', \'the encounter is ...\')")\n\n    def conseq(self) -> Atom:\n        if self.at("the candidate code of"):\n            self.phrase("the candidate code of")\n            v = self.var()\n            self.phrase("is assigned")\n            return Atom("assign_candidate", (v,))\n        if self.at("the codes"):\n            self.phrase("the codes")\n            a = self.code()\n            self.word("and")\n            b = self.code()\n            self.word("conflict")\n            return Atom("conflict", (a, b))\n        self.phrase("the code")\n        c = self.code_or_var()\n        if self.peek().lower() == "precedes":\n            self.phrase("precedes the code")\n            return Atom("precedes", (c, self.code_or_var()))\n        if isinstance(c, Var):\n            self.fail("\'precedes\' (only sequencing may use a code variable)")\n        self.word("is")\n        if self.peek().lower() == "not":\n            self.i += 1\n            self.word("assigned")\n            return Atom("block", (c,))\n        self.word("assigned")\n        return Atom("assign", (c,))\n\n    def rule(self, rid: str) -> AceRule:\n        self.word("if")\n        conds = [self.cond()]\n        while self.peek().lower() == "and":\n            self.i += 1\n            conds.append(self.cond())\n        self.word("then")\n        conseqs = [self.conseq()]\n        while self.peek().lower() == "and":\n            self.i += 1\n            conseqs.append(self.conseq())\n        self.word(".")\n        if self.i != len(self.toks):\n            self.fail("end of sentence")\n        return AceRule(rid, self.s, conds, conseqs)\n\n\ndef parse_rule(sentence: str, rid: str = "r1", concepts: Optional[Iterable[str]] = None, line: Optional[int] = None) -> AceRule:\n    return _Parser(sentence, set(concepts) if concepts is not None else None, line).rule(rid)\n\n\ndef parse_rules(text: str, concepts: Optional[Iterable[str]] = None) -> List[AceRule]:\n    """Parse a rules file: one sentence per line; \'#\' starts a comment; an optional\n    \'ID:\' prefix names the rule (otherwise r1, r2, ...)."""\n    cs = set(concepts) if concepts is not None else None\n    out, n = [], 0\n    for ln, raw in enumerate(text.splitlines(), 1):\n        line = raw.split("#", 1)[0].strip()\n        if not line:\n            continue\n        n += 1\n        rid = f"r{n}"\n        m = re.match(r"^([A-Za-z][\\w.-]*):\\s+(.*)$", line)\n        if m:\n            rid, line = m.group(1), m.group(2)\n        out.append(parse_rule(line, rid, cs, ln))\n    ids = [r.rid for r in out]\n    dup = {i for i in ids if ids.count(i) > 1}\n    if dup:\n        raise ValueError(f"duplicate rule ids: {sorted(dup)}")\n    return out\n\n\ndef compile_rules(rules: Sequence[AceRule]) -> Program:\n    prog = Program()\n    for r in rules:\n        for d in r.to_datalog():\n            prog.add(d)\n    prog.add(Rule(FINAL_RULE.head, list(FINAL_RULE.body), FINAL_RULE.rid, FINAL_RULE.source))\n    prog.strata()  # raises DatalogError if the rules are not stratifiable\n    return prog\n',
    'yappu_fsm/rules/coder.py': '"""Note -> linker facts -> Datalog (compiled from controlled-English rules) -> codes with proofs."""\n\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence, Tuple, Union\n\nfrom ..clinical.linker import ClinicalLinker, Finding\nfrom .ace import AceRule, compile_rules, parse_rules\nfrom .datalog import Fact, Program\n\nPATIENT = "patient"\nSIDE = {"RIGHT": "right", "LEFT": "left", "BILATERAL": "both"}\nPRED = {"PRESENT": "has", "UNCERTAIN": "uncertain", "HISTORICAL": "history",\n        "FAMILY_HISTORY": "family", "NEGATED": "negated", "HYPOTHETICAL": "hypothetical"}\n\n\ndef concept(entity: str) -> str:\n    return entity.replace(" ", "-")\n\n\ndef code_kind(code: str) -> str:\n    """ICD-10-CM chapter class: R = symptom, Z = factor, V-Y = external cause, else diagnosis."""\n    c = code[:1].upper()\n    return "symptom" if c == "R" else "factor" if c == "Z" else "external" if c in "VWXY" else "diagnosis"\n\n\n@dataclass\nclass CodingResult:\n    codes: List[str]                                  # final, sequenced\n    blocked: Dict[str, List[str]]                     # code -> proof lines of the block\n    conflicts: List[Tuple[str, str, List[str]]]       # (code, code, proof)\n    proofs: Dict[str, List[str]]                      # final code -> proof lines\n    findings: List[Finding]\n    facts: List[Fact]\n    setting: str\n\n    def report(self) -> str:\n        out = [f"setting: {self.setting}", "final codes: " + (", ".join(self.codes) or "(none)")]\n        for c in self.codes:\n            out.append(f"\\n[{c}]")\n            out += ["  " + l for l in self.proofs[c]]\n        for c, proof in self.blocked.items():\n            out.append(f"\\nblocked {c}:")\n            out += ["  " + l for l in proof]\n        for a, b, proof in self.conflicts:\n            out.append(f"\\nCONFLICT {a} / {b} (coder review):")\n            out += ["  " + l for l in proof]\n        return "\\n".join(out)\n\n\nclass RuleCoder:\n    """Combine the fast rule-based linker (facts) with controlled-English coding rules (Datalog).\n\n    The linker always runs in outpatient mode (uncertain diagnoses produce no code);\n    the encounter setting becomes the fact ``setting(patient, outpatient|inpatient)``\n    and rules such as INPT_UNCERTAIN decide what to do with uncertain findings.\n    """\n\n    def __init__(self, linker: ClinicalLinker, rules: Union[str, Path, Sequence[AceRule]]):\n        self.linker = linker if linker.setting == "outpatient" else ClinicalLinker(linker.lexicon)\n        if isinstance(rules, (str, Path)) and Path(str(rules)).exists():\n            rules = Path(rules).read_text(encoding="utf-8")\n        if isinstance(rules, str):\n            rules = parse_rules(rules, concepts=self.linker.lexicon.keys())\n        self.rules: List[AceRule] = list(rules)\n        self.program: Program = compile_rules(self.rules)\n        self.rules_by_id = {r.rid: r for r in self.program.rules}\n        # every code the system can mention gets a kind/3 fact\n        known = {c for v in self.linker.lexicon.values() for x in v.values() for c in ([x] if isinstance(x, str) else x)}\n        for r in self.program.rules:\n            for lit in [r.head] + r.body:\n                known |= {a for a in lit.args if isinstance(a, str) and re.match(r"^[A-Z][0-9][0-9A-Z]", a)}\n        self.kind_facts: List[Fact] = [("kind", PATIENT, c, code_kind(c)) for c in sorted(known)]\n\n    def _candidates(self, f: Finding) -> List[str]:\n        cmap = self.linker.lexicon.get(f.entity, {})\n        val = cmap.get(f"PRESENT_{f.laterality}") or cmap.get("PRESENT_UNSPECIFIED") or cmap.get("PRESENT")\n        return [] if val is None else ([val] if isinstance(val, str) else list(val))\n\n    def facts_for(self, findings: Sequence[Finding], setting: str) -> List[Fact]:\n        facts: List[Fact] = [("patient", PATIENT), ("setting", PATIENT, setting)]\n        for f in findings:\n            c = concept(f.entity)\n            facts.append((PRED[f.assertion], PATIENT, c))\n            if f.laterality in SIDE:\n                facts.append(("side", PATIENT, c, SIDE[f.laterality]))\n            for code in self._candidates(f):\n                facts.append(("candidate", PATIENT, c, code))\n            for code in f.codes:                     # codes the linker assigned directly\n                facts.append(("code", PATIENT, code))\n        seen, out = set(), []\n        for x in facts + getattr(self, "kind_facts", []):\n            if x not in seen:\n                seen.add(x)\n                out.append(x)\n        return out\n\n    def code(self, note: str, setting: str = "outpatient") -> CodingResult:\n        if setting not in ("outpatient", "inpatient"):\n            raise ValueError("setting must be \'outpatient\' or \'inpatient\'")\n        findings = self.linker.process(note)\n        facts = self.facts_for(findings, setting)\n        m = self.program.run(facts)\n        explain = lambda f: m.explain(f, self.rules_by_id, hide=("patient",))\n\n        final_codes = [f[2] for f in m.query("final")]\n        final = _sequence(self._base_order(final_codes, findings, m), [(f[2], f[3]) for f in m.query("precedes")])\n        blocked = {f[2]: explain(f) for f in m.query("blocked") if ("code", PATIENT, f[2]) in m.facts}\n        conflicts = [(f[2], f[3], explain(f)) for f in m.query("conflict")]\n        proofs = {c: explain(("final", PATIENT, c)) for c in final}\n        return CodingResult(final, blocked, conflicts, proofs, findings, facts, setting)\n\n\n    def _base_order(self, codes: List[str], findings: Sequence[Finding], m) -> List[str]:\n        """Default position of each final code before \'precedes\' constraints are applied.\n\n        * a code the linker produced keeps the position of its finding;\n        * a rule-derived code takes the position of the code it replaced (a code blocked\n          by the same rule, or blocked because of it), e.g. E11.22 where E11.9 was;\n        * otherwise it sits at the earliest finding its proof mentions.\n        """\n        key: Dict[str, Tuple[int, int]] = {}\n        concept_pos: Dict[str, int] = {}\n        for i, f in enumerate(findings):\n            concept_pos.setdefault(concept(f.entity), i)\n            for c in f.codes:\n                key.setdefault(c, (i, len(key)))\n        given = set(key)\n        base = lambda rid: rid.split(".")[0]\n        for c in codes:\n            if c in given:\n                continue\n            why = m.why.get(("code", PATIENT, c), ("?", ()))\n            links = []\n            for bf in m.query("blocked"):\n                bwhy = m.why.get(bf, ("?", ()))\n                if bf[2] in key and (base(bwhy[0]) == base(why[0]) or ("code", PATIENT, c) in bwhy[1]):\n                    links.append(key[bf[2]])\n            if links:\n                key[c] = (min(links)[0], min(links)[1] - 0.5)\n                continue\n            pos = [concept_pos[u[2]] for u in _walk(m, ("code", PATIENT, c)) if len(u) > 2 and u[2] in concept_pos]\n            key[c] = (min(pos), -1) if pos else (10 ** 6, len(key))\n        return sorted(codes, key=lambda c: key.get(c, (10 ** 6, 0)))\n\n\ndef _walk(m, fact, seen=None):\n    seen = set() if seen is None else seen\n    if fact in seen:\n        return []\n    seen.add(fact)\n    out = [fact]\n    for u in m.why.get(fact, ("?", ()))[1]:\n        if u[0] != "not":\n            out += _walk(m, u, seen)\n    return out\n\n\ndef _sequence(codes: List[str], before: List[Tuple[str, str]]) -> List[str]:\n    """Stable topological order honouring \'X precedes Y\' where both are present."""\n    present = set(codes)\n    edges = [(a, b) for a, b in before if a in present and b in present and a != b]\n    out, remaining = [], list(codes)\n    while remaining:\n        for c in remaining:\n            if not any(b == c and a in remaining for a, b in edges):\n                out.append(c)\n                remaining.remove(c)\n                break\n        else:  # cycle: keep original order for the rest\n            out += remaining\n            break\n    return out\n',
    'yappu_fsm/rules/datalog.py': '"""A small Datalog engine: stratified negation, semi-naive evaluation, provenance.\n\nTerms are plain strings; variables are :class:`Var` objects.  Facts are tuples\n``(predicate, arg1, arg2, ...)``.  Every derived fact remembers the rule and the\nground body facts that first produced it, so :meth:`Program.explain` can print\na proof tree.\n\nDatalog (Horn clauses without function symbols, with stratified negation) always\nterminates and runs in polynomial time in the number of facts – unlike general\nfirst-order theorem proving, which is only semi-decidable.\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom collections import defaultdict\nfrom dataclasses import dataclass, field\nfrom typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple\n\nFact = Tuple[str, ...]\n\n\n@dataclass(frozen=True)\nclass Var:\n    name: str\n\n    def __repr__(self) -> str:\n        return self.name\n\n\n@dataclass(frozen=True)\nclass Literal:\n    pred: str\n    args: Tuple[object, ...]\n    negated: bool = False\n\n    def vars(self) -> Set[Var]:\n        return {a for a in self.args if isinstance(a, Var)}\n\n    def __str__(self) -> str:\n        s = f"{self.pred}({\', \'.join(_fmt(a) for a in self.args)})"\n        return ("not " + s) if self.negated else s\n\n\n@dataclass\nclass Rule:\n    head: Literal\n    body: List[Literal]\n    rid: str = ""\n    source: str = ""          # original ACE sentence, if any\n\n    def __str__(self) -> str:\n        return f"{self.head} :- {\', \'.join(map(str, self.body))}." if self.body else f"{self.head}."\n\n\ndef _fmt(a) -> str:\n    if isinstance(a, Var):\n        return a.name\n    s = str(a)\n    return s if re.fullmatch(r"[a-z][a-z0-9_]*", s) else \'"\' + s.replace(\'"\', \'\\\\"\') + \'"\'\n\n\nclass DatalogError(ValueError):\n    pass\n\n\nclass Program:\n    def __init__(self, rules: Iterable[Rule] = ()):\n        self.rules: List[Rule] = []\n        for r in rules:\n            self.add(r)\n\n    def add(self, rule: Rule):\n        if rule.head.negated:\n            raise DatalogError(f"{rule.rid}: negated head")\n        positive = set().union(*[l.vars() for l in rule.body if not l.negated]) if rule.body else set()\n        unsafe = (rule.head.vars() | set().union(*[l.vars() for l in rule.body if l.negated] or [set()])) - positive\n        if unsafe:\n            raise DatalogError(f"{rule.rid}: unsafe variable(s) {sorted(v.name for v in unsafe)} "\n                               f"must appear in a positive condition")\n        # evaluate positive literals first, negations last (safe rules make this sound)\n        rule.body = [l for l in rule.body if not l.negated] + [l for l in rule.body if l.negated]\n        self.rules.append(rule)\n\n    # ------------------------------------------------------------ strata\n    def strata(self) -> List[List[Rule]]:\n        preds = {r.head.pred for r in self.rules} | {l.pred for r in self.rules for l in r.body}\n        level = {p: 0 for p in preds}\n        n = len(preds)\n        changed = True\n        while changed:\n            changed = False\n            for r in self.rules:\n                for l in r.body:\n                    need = level[l.pred] + (1 if l.negated else 0)\n                    if level[r.head.pred] < need:\n                        level[r.head.pred] = need\n                        changed = True\n                        if need > n:\n                            raise DatalogError(f"not stratifiable: \'{r.head.pred}\' depends negatively on itself "\n                                               f"(via rule {r.rid})")\n        out: Dict[int, List[Rule]] = defaultdict(list)\n        for r in self.rules:\n            out[level[r.head.pred]].append(r)\n        return [out[k] for k in sorted(out)]\n\n    # ------------------------------------------------------------ evaluation\n    def run(self, facts: Iterable[Fact]) -> "Model":\n        model = Model()\n        for f in facts:\n            model.add(tuple(f), ("fact", ()))\n        for stratum in self.strata():\n            self._fixpoint(stratum, model)\n        return model\n\n    def _fixpoint(self, rules: List[Rule], m: "Model"):\n        delta: Set[Fact] = set(m.facts)          # first round: everything is new\n        while delta:\n            new: Set[Fact] = set()\n            delta_by_pred = defaultdict(list)\n            for f in delta:\n                delta_by_pred[f[0]].append(f)\n            for r in rules:\n                pos = [i for i, l in enumerate(r.body) if not l.negated]\n                # semi-naive: at least one positive literal must bind to a delta fact\n                for di in pos:\n                    if r.body[di].pred not in delta_by_pred:\n                        continue\n                    for binding, used in self._match(r.body, 0, {}, [], m, di, delta_by_pred):\n                        head = (r.head.pred,) + tuple(binding[a] if isinstance(a, Var) else a for a in r.head.args)\n                        if head not in m.facts and head not in new:\n                            new.add(head)\n                            m.add(head, (r.rid, tuple(used)))\n                if not pos and r.body == []:\n                    head = (r.head.pred,) + tuple(r.head.args)\n                    if head not in m.facts:\n                        new.add(head)\n                        m.add(head, (r.rid, ()))\n            delta = new\n\n    def _match(self, body, i, binding, used, m, delta_i, delta_by_pred):\n        """Enumerate bindings for body[i:]; literal ``delta_i`` reads only new facts."""\n        if i == len(body):\n            yield binding, used\n            return\n        lit = body[i]\n        if lit.negated:  # all variables are bound here (positives come first)\n            ground = (lit.pred,) + tuple(binding[a] if isinstance(a, Var) else a for a in lit.args)\n            if ground not in m.facts:\n                yield from self._match(body, i + 1, binding, used + [("not",) + ground], m, delta_i, delta_by_pred)\n            return\n        source = delta_by_pred.get(lit.pred, []) if i == delta_i else m.by_pred.get(lit.pred, [])\n        for f in source:\n            if len(f) != len(lit.args) + 1:\n                continue\n            b = dict(binding)\n            for a, v in zip(lit.args, f[1:]):\n                if isinstance(a, Var):\n                    if b.setdefault(a, v) != v:\n                        break\n                elif a != v:\n                    break\n            else:\n                yield from self._match(body, i + 1, b, used + [f], m, delta_i, delta_by_pred)\n\n    def to_text(self) -> str:\n        lines = []\n        for r in self.rules:\n            if r.source:\n                lines.append(f"% {r.rid}: {r.source}")\n            lines.append(str(r))\n        return "\\n".join(lines)\n\n\nclass Model:\n    def __init__(self):\n        self.facts: Set[Fact] = set()\n        self.by_pred: Dict[str, List[Fact]] = defaultdict(list)\n        self.why: Dict[Fact, Tuple[str, tuple]] = {}\n\n    def add(self, f: Fact, why):\n        self.facts.add(f)\n        self.by_pred[f[0]].append(f)\n        self.why[f] = why\n\n    def query(self, pred: str) -> List[Fact]:\n        return list(self.by_pred.get(pred, []))\n\n    def explain(self, f: Fact, rules_by_id: Optional[Dict[str, Rule]] = None, depth: int = 0, seen=None,\n                hide: Sequence[str] = ()) -> List[str]:\n        seen = set() if seen is None else seen\n        pad = "  " * depth\n        if f[0] == "not":\n            return [f"{pad}not {f[1]}({\', \'.join(f[2:])})  [absent]"]\n        rid, used = self.why.get(f, ("?", ()))\n        label = "given" if rid == "fact" else rid\n        src = ""\n        if rules_by_id and rid in rules_by_id and rules_by_id[rid].source:\n            src = f"  « {rules_by_id[rid].source} »"\n        out = [f"{pad}{f[0]}({\', \'.join(f[1:])})  [{label}]{src}"]\n        if f in seen:\n            return out\n        seen.add(f)\n        for u in used:\n            if u[0] in hide:\n                continue\n            out += self.explain(u, rules_by_id, depth + 1, seen, hide)\n        return out\n',
    'yappu_fsm/server.py': '"""OpenAI-compatible HTTP endpoint (stdlib only).\n\n    POST /v1/chat/completions   {"model": "...", "messages": [...]}\n    GET  /v1/models\n\nModels:\n  * ``yappu-icd-linker``       – runs the clinical linker on the last user message;\n                                  the assistant content is a JSON list of findings.\n                                  Optional body field ``"setting": "inpatient"`` applies the\n                                  ICD-10-CM inpatient rule for uncertain diagnoses.\n  * ``yappu-icd-rules``        – linker facts + controlled-English coding rules (Datalog):\n                                  final sequenced codes, blocked codes, conflicts, proofs.\n  * ``yappu-venba-validator``  – validates the last user message as a Nerisai Venba;\n                                  the assistant content is the text report.\n\nAny OpenAI client works, e.g.::\n\n    from openai import OpenAI\n    c = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")\n    c.chat.completions.create(model="yappu-icd-linker",\n                              messages=[{"role": "user", "content": "Denies SOB, has HTN."}])\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport time\nimport uuid\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom typing import Optional\n\nfrom . import data_path\nfrom .clinical import ClinicalLinker, load_lexicon\nfrom .prosody import validate_nerisai_venba\n\nMODELS = ("yappu-icd-linker", "yappu-icd-rules", "yappu-venba-validator")\n\n\ndef _last_user_text(messages) -> str:\n    for m in reversed(messages or []):\n        if m.get("role") == "user":\n            c = m.get("content", "")\n            if isinstance(c, list):  # content parts\n                c = " ".join(p.get("text", "") for p in c if isinstance(p, dict))\n            return c\n    return ""\n\n\n_CODERS: dict = {}\n\n\ndef _rule_coder(linker: ClinicalLinker):\n    from .rules import RuleCoder\n    key = id(linker)\n    if key not in _CODERS:\n        _CODERS[key] = RuleCoder(linker, data_path("icd10cm_rules.ace"))\n    return _CODERS[key]\n\n\ndef complete(body: dict, linker: ClinicalLinker) -> dict:\n    model = body.get("model", MODELS[0])\n    if model not in MODELS:\n        raise ValueError(f"unknown model \'{model}\'; available: {\', \'.join(MODELS)}")\n    text = _last_user_text(body.get("messages"))\n    if model == "yappu-icd-linker":\n        setting = body.get("setting", "outpatient")   # extra field: "outpatient" | "inpatient"\n        if setting != linker.setting:\n            linker = ClinicalLinker(linker.lexicon, setting=setting)\n        found = linker.process(text)\n        content = json.dumps({"findings": [f.to_dict() for f in found],\n                              "allowed_codes": ClinicalLinker.allowed_codes(found)}, ensure_ascii=False)\n    elif model == "yappu-icd-rules":\n        res = _rule_coder(linker).code(text, body.get("setting", "outpatient"))\n        content = json.dumps({"codes": res.codes, "blocked": list(res.blocked),\n                              "conflicts": [[a, b] for a, b, _ in res.conflicts], "proofs": res.proofs},\n                             ensure_ascii=False)\n    else:\n        content = "\\n".join(validate_nerisai_venba(text).lines())\n    return {\n        "id": "chatcmpl-" + uuid.uuid4().hex[:24],\n        "object": "chat.completion",\n        "created": int(time.time()),\n        "model": model,\n        "choices": [{"index": 0, "message": {"role": "assistant", "content": content}, "finish_reason": "stop"}],\n        "usage": {"prompt_tokens": len(text.split()), "completion_tokens": len(content.split()),\n                  "total_tokens": len(text.split()) + len(content.split())},\n    }\n\n\ndef make_handler(linker: ClinicalLinker):\n    class Handler(BaseHTTPRequestHandler):\n        def _send(self, code: int, payload: dict):\n            data = json.dumps(payload, ensure_ascii=False).encode("utf-8")\n            self.send_response(code)\n            self.send_header("Content-Type", "application/json; charset=utf-8")\n            self.send_header("Content-Length", str(len(data)))\n            self.end_headers()\n            self.wfile.write(data)\n\n        def do_GET(self):\n            if self.path.rstrip("/") == "/v1/models":\n                self._send(200, {"object": "list", "data": [\n                    {"id": m, "object": "model", "owned_by": "yappu-fsm"} for m in MODELS]})\n            else:\n                self._send(404, {"error": {"message": "not found"}})\n\n        def do_POST(self):\n            if self.path.rstrip("/") != "/v1/chat/completions":\n                return self._send(404, {"error": {"message": "not found"}})\n            try:\n                n = int(self.headers.get("Content-Length", 0))\n                body = json.loads(self.rfile.read(n) or b"{}")\n                if body.get("stream"):\n                    raise ValueError("streaming is not supported")\n                self._send(200, complete(body, linker))\n            except ValueError as e:\n                self._send(400, {"error": {"message": str(e), "type": "invalid_request_error"}})\n\n        def log_message(self, *args):  # quiet\n            pass\n\n    return Handler\n\n\ndef serve(host: str = "127.0.0.1", port: int = 8000, lexicon_path: Optional[str] = None) -> ThreadingHTTPServer:\n    linker = ClinicalLinker(load_lexicon(lexicon_path or data_path("icd10cm_example_lexicon.json")))\n    return ThreadingHTTPServer((host, port), make_handler(linker))\n',
    'yappu_fsm/tamil.py': '"""Tamil orthography and prosody primitives.\n\nEverything here is pure Python (no third-party dependencies):\n\n* ``graphemes``       - split a word into visual letters (consonant + vowel sign / pulli).\n* ``letter_classes``  - classify letters as Kuril (K), Nedil (N) or Otru (O).\n* ``encode_asai``     - reduce a word to its binary Asai string (\'1\' = Ner, \'0\' = Nirai).\n* ``ethukai_key``     - 2nd-letter rhyme key (consonant of the 2nd letter + length of the 1st).\n* ``monai_key``       - 1st-letter alliteration class using the traditional Ina Ezhuthu groups.\n* ``encode_english``  - rough English syllable-weight heuristic (approximate; see docstring).\n"""\n\nfrom __future__ import annotations\n\nimport re\nimport unicodedata\nfrom typing import List, Optional, Tuple\n\nPULLI = "்"  # ்\nAU_LENGTH_MARK = "ௗ"\nAYTHAM = "ஃ"  # ஃ\n\n# Dependent vowel signs\nNEDIL_SIGNS = {"ா", "ீ", "ூ", "ே", "ை", "ோ", "ௌ"}  # ா ீ ூ ே ை ோ ௌ\nKURIL_SIGNS = {"ி", "ு", "ெ", "ொ"}  # ி ு ெ ொ\nCOMBINING = NEDIL_SIGNS | KURIL_SIGNS | {PULLI, AU_LENGTH_MARK}\n\nINDEPENDENT_KURIL = {"அ", "இ", "உ", "எ", "ஒ"}\nINDEPENDENT_NEDIL = {"ஆ", "ஈ", "ஊ", "ஏ", "ஐ", "ஓ", "ஔ"}\n\n# Hard consonants (வல்லினம்) – used for kutriyalukaram (Kaasu / Pirappu endings)\nVALLINAM = {"க", "ச", "ட", "த", "ப", "ற"}\n\nTAMIL_RE = re.compile(r"[\u0b80-\u0bff]")\n\n\ndef is_consonant(ch: str) -> bool:\n    return "க" <= ch <= "ஹ"\n\n\ndef normalize(text: str) -> str:\n    """NFC-normalise so decomposed ொ/ோ/ௌ become single code points."""\n    return unicodedata.normalize("NFC", text)\n\n\ndef is_tamil(text: str) -> bool:\n    return bool(TAMIL_RE.search(text))\n\n\ndef graphemes(word: str) -> List[str]:\n    """Split a Tamil word into visual letters.\n\n    >>> graphemes("வண்ண")\n    [\'வ\', \'ண்\', \'ண\']\n    """\n    letters: List[str] = []\n    for ch in normalize(word):\n        if ch in COMBINING and letters:\n            letters[-1] += ch\n        elif TAMIL_RE.match(ch):\n            letters.append(ch)\n    return letters\n\n\ndef letter_class(letter: str) -> Optional[str]:\n    """Return \'K\' (kuril), \'N\' (nedil), \'O\' (otru / aytham) or None."""\n    if not letter:\n        return None\n    base = letter[0]\n    if base in INDEPENDENT_KURIL:\n        return "K"\n    if base in INDEPENDENT_NEDIL:\n        return "N"\n    if base == AYTHAM:\n        return "O"\n    if is_consonant(base):\n        sign = letter[1:2]\n        if sign == PULLI:\n            return "O"\n        if sign in NEDIL_SIGNS:\n            return "N"\n        return "K"  # bare consonant (inherent அ) or short vowel sign\n    return None\n\n\ndef letter_classes(word: str) -> List[str]:\n    return [c for c in (letter_class(g) for g in graphemes(word)) if c]\n\n\ndef asai_units(word: str) -> List[Tuple[str, str]]:\n    """Split a word into Asai units, returning (text, \'1\'|\'0\') pairs.\n\n    Rules (யாப்பிலக்கணம்):\n      * a Nedil always closes its own Ner asai;\n      * a Kuril followed by a Kuril or a Nedil forms a Nirai asai;\n      * a lone Kuril forms a Ner asai;\n      * trailing Otru letters attach to the preceding asai.\n    """\n    gs = graphemes(word)\n    cls = [letter_class(g) for g in gs]\n    units: List[Tuple[str, str]] = []\n    i, n = 0, len(gs)\n    while i < n:\n        c = cls[i]\n        if c == "O":  # stray otru at word start – attach to previous or skip\n            if units:\n                units[-1] = (units[-1][0] + gs[i], units[-1][1])\n            i += 1\n            continue\n        if c == "K" and i + 1 < n and cls[i + 1] in ("K", "N"):\n            text, kind, i = gs[i] + gs[i + 1], "0", i + 2\n        else:\n            text, kind, i = gs[i], "1", i + 1\n        while i < n and cls[i] == "O":\n            text += gs[i]\n            i += 1\n        units.append((text, kind))\n    return units\n\n\ndef encode_asai(word: str) -> str:\n    """Binary Asai string for a Tamil word.\n\n    >>> encode_asai("கூடினால்")   # கூ / டினால்  -> Koovilam\n    \'10\'\n    """\n    return "".join(kind for _, kind in asai_units(word))\n\n\ndef split_display(word: str) -> str:\n    """Human-readable asai split, e.g. \'கூ/டினால்\'."""\n    return "/".join(t for t, _ in asai_units(word))\n\n\ndef has_kutriyalukaram_ending(word: str) -> bool:\n    """True when the word ends in a short final உ on a hard consonant (கு சு டு து பு று)."""\n    gs = graphemes(word)\n    if len(gs) < 2:\n        return False\n    last = gs[-1]\n    return len(last) == 2 and last[0] in VALLINAM and last[1] == "ு"\n\n\n# --------------------------------------------------------------------------\n# Ethukai (எதுகை) – 2nd-letter rhyme\n# --------------------------------------------------------------------------\n\n\ndef ethukai_key(word: str, strict: bool = False) -> Optional[str]:\n    """Key used to compare Ethukai between line-initial words.\n\n    Default (traditional) rule: the 2nd letters share the same consonant and the\n    1st letters have the same length (both kuril or both nedil).  ``strict=True``\n    compares the full 2nd letter instead of its consonant.\n    """\n    gs = graphemes(word)\n    if len(gs) < 2:\n        return None\n    first_len = "N" if letter_class(gs[0]) == "N" else "K"\n    second = gs[1] if strict else gs[1][0]\n    return f"{first_len}:{second}"\n\n\ndef ethukai_match(a: str, b: str, strict: bool = False) -> bool:\n    ka, kb = ethukai_key(a, strict), ethukai_key(b, strict)\n    return ka is not None and ka == kb\n\n\n# --------------------------------------------------------------------------\n# Monai (மோனை) – 1st-letter alliteration with Ina Ezhuthu equivalence\n# --------------------------------------------------------------------------\n\n_VOWEL_GROUP = {\n    "அ": "A", "ஆ": "A", "ஐ": "A", "ஔ": "A",\n    "இ": "I", "ஈ": "I", "எ": "I", "ஏ": "I",\n    "உ": "U", "ஊ": "U", "ஒ": "U", "ஓ": "U",\n}\n_SIGN_GROUP = {\n    "": "A", "ா": "A", "ை": "A", "ௌ": "A",\n    "ி": "I", "ீ": "I", "ெ": "I", "ே": "I",\n    "ு": "U", "ூ": "U", "ொ": "U", "ோ": "U",\n}\n# Traditional consonant Monai pairs: ச/த, ஞ/ந, ம/வ\n_CONSONANT_GROUP = {"ச": "சத", "த": "சத", "ஞ": "ஞந", "ந": "ஞந", "ம": "மவ", "வ": "மவ"}\n\n\ndef monai_key(word: str) -> Optional[str]:\n    if not is_tamil(word):\n        m = re.search(r"[a-zA-Z]", word)\n        return m.group(0).lower() if m else None\n    gs = graphemes(word)\n    if not gs:\n        return None\n    first = gs[0]\n    if first[0] in _VOWEL_GROUP:\n        return "V:" + _VOWEL_GROUP[first[0]]\n    if is_consonant(first[0]):\n        cons = _CONSONANT_GROUP.get(first[0], first[0])\n        return f"C:{cons}:{_SIGN_GROUP.get(first[1:2], \'A\')}"\n    return None\n\n\n# --------------------------------------------------------------------------\n# English (approximate)\n# --------------------------------------------------------------------------\n\n\ndef encode_english(word: str) -> str:\n    """Very rough English syllable-weight heuristic mapped onto Ner/Nirai.\n\n    Heavy syllables (long vowel digraph or ≥4 letters) -> \'1\', light -> \'0\';\n    runs of light syllables collapse to one \'0\'.  This is an illustration of the\n    idea only; real English meter needs a pronouncing dictionary (e.g. CMUdict).\n    """\n    clean = re.sub(r"[^a-zA-Z]", "", word).lower()\n    if not clean:\n        return ""\n    syllables = re.findall(r"[^aeiouy]*[aeiouy]+(?:[^aeiouy](?![aeiouy]))*", clean) or [clean]\n    weights = ["1" if re.search(r"[aeiou]{2}|[aeiou][a-z]e$", s) or len(s) >= 4 else "0" for s in syllables]\n    return re.sub(r"00+", "0", "".join(weights))[:3]\n\n\ndef encode(word: str) -> str:\n    """Asai for Tamil words; approximate weight pattern for English words."""\n    return encode_asai(word) if is_tamil(word) else encode_english(word)\n',
    'tests/conftest.py': 'import pytest\n\nfrom yappu_fsm import data_path\nfrom yappu_fsm.clinical import ClinicalLinker, load_lexicon\n\nPOEM_1 = """எண்ணும் எழுத்தும் இசையோடு கூடினால்\nபண்ணும் மணியாகும் பார்ப்பவர் - கண்ணும்\nஅமுதத் தமிழில் அமைந்த இலக்கணம்\nநம்மொழி யாக்கும் இசை."""\nPOEM_2 = """வண்ண நிலவொளி வானில் தவழ்ந்திட\nஎண்ணற்ற விண்மீன்கள் ஏற்றிடும் - விண்ணில்\nஇருளினை நீக்கும் இனிய ஒளியால்\nமருளினை மாற்றும் மதி."""\nNANNERI = """அடக்க முடையா ரறிவிலரென் றெண்ணிக்\nகடக்கக் கருதவும் வேண்டா - மடைத்தலையில்\nஓடுமீ னோட உறுமீன் வருமளவும்\nவாடி யிருக்குமாம் கொக்கு."""\n\n\n@pytest.fixture(scope="session")\ndef lexicon():\n    return load_lexicon(data_path("icd10cm_example_lexicon.json"))\n\n\n@pytest.fixture\ndef linker(lexicon):\n    return ClinicalLinker(lexicon)\n',
    'tests/test_clinical.py': 'import pytest\n\nfrom yappu_fsm.clinical import clinical_phonetic_key\n\n\ndef by_entity(findings):\n    return {f.entity: f for f in findings}\n\n\n# ---- laterality ---------------------------------------------------------------\n\ndef test_multi_laterality_no_cross_contamination(linker):\n    r = by_entity(linker.process("Pt presents with mild NPDR OD and lt knee osteoarthritis."))\n    assert r["mild nonproliferative diabetic retinopathy"].laterality == "RIGHT"\n    assert r["mild nonproliferative diabetic retinopathy"].codes == ["E11.3291"]\n    assert r["knee osteoarthritis"].laterality == "LEFT"\n    assert r["knee osteoarthritis"].codes == ["M17.12"]\n\n\ndef test_pre_and_post_laterality(linker):\n    r = by_entity(linker.process("Right hip pain noted along with mild NPDR OS."))\n    assert r["hip pain"].codes == ["M25.551"]\n    assert r["mild nonproliferative diabetic retinopathy"].codes == ["E11.3292"]\n\n\ndef test_bilateral(linker):\n    r = by_entity(linker.process("Exam shows b/l knee osteoarthritis and mild NPDR OU."))\n    assert r["knee osteoarthritis"].codes == ["M17.0"]\n    assert r["mild nonproliferative diabetic retinopathy"].codes == ["E11.3293"]\n\n\ndef test_bilateral_hip_pain_codes_both_sides(linker):\n    (f,) = linker.process("Bilateral hip pain worse on stairs.")\n    assert f.codes == ["M25.551", "M25.552"]\n\n\ndef test_laterality_distance_threshold(linker):\n    (f,) = linker.process("Right arm scratch noted on skin exam yesterday and patient has hypertension.")\n    assert f.entity == "hypertension" and f.laterality == "UNSPECIFIED" and f.codes == ["I10"]\n\n\n# ---- assertion ----------------------------------------------------------------\n\ndef test_wall_resets_negation(linker):\n    r = by_entity(linker.process("Patient denies SOB, but has uncontrolled HTN."))\n    assert r["shortness of breath"].status == "DO_NOT_CODE"\n    assert r["hypertension"].codes == ["I10"]\n\n\ndef test_nearest_trigger_wins(linker):\n    # the thread\'s version marked SOB as HISTORICAL here\n    r = by_entity(linker.process("Denies SOB, history of MI."))\n    assert r["shortness of breath"].assertion == "NEGATED"\n    assert r["myocardial infarction"].assertion == "HISTORICAL"\n    assert r["myocardial infarction"].codes == ["I25.2"]\n\n\ndef test_trigger_scope_limit(linker):\n    (f,) = linker.process("No known drug allergies, takes lisinopril for longstanding hypertension.")\n    assert f.assertion == "PRESENT"\n\n\ndef test_post_triggers(linker):\n    (f,) = linker.process("Chest X-ray clear; pneumonia ruled out.")\n    assert f.assertion == "NEGATED"\n    (g,) = linker.process("Pneumonia has resolved after treatment.")\n    assert g.assertion == "HISTORICAL"\n\n\ndef test_uncertain_and_family_history(linker):\n    (f,) = linker.process("R/O CAP.")\n    assert f.assertion == "UNCERTAIN" and f.status == "DO_NOT_CODE"\n    r = by_entity(linker.process("Family history of HTN. Mother had type 2 diabetes."))\n    assert r["hypertension"].codes == ["Z82.49"]\n    assert r["type 2 diabetes"].codes == ["Z83.3"]\n\n\ndef test_negated_with_laterality_still_blocked(linker):\n    (f,) = linker.process("Patient denies right hip pain today.")\n    assert f.laterality == "RIGHT" and f.status == "DO_NOT_CODE"\n\n\ndef test_pseudo_trigger(linker):\n    (f,) = linker.process("No change in hypertension since last visit.")\n    assert f.assertion == "PRESENT"\n\n\ndef test_case_sensitive_abbreviations(linker):\n    assert linker.process("Bruise over the knee cap.") == []\n    assert linker.process("Possible CAP.")[0].entity == "pneumonia"\n\n\ndef test_decimal_does_not_split_clause(linker):\n    assert len(linker.split_clauses("HbA1c 7.5 today. Denies SOB")) == 2\n\n\n# ---- fuzzy matching -----------------------------------------------------------\n\n@pytest.mark.parametrize("a,b", [("osteoarthritis", "osteoarthritus"), ("pneumonia", "pnemonia"),\n                                 ("nephropathy", "nefropathy"), ("hypertension", "hypertensn"),\n                                 ("retinopathy", "retinopthy")])\ndef test_phonetic_keys(a, b):\n    assert clinical_phonetic_key(a) == clinical_phonetic_key(b)\n\n\ndef test_typo_recovery_end_to_end(linker):\n    r = by_entity(linker.process(\n        "Pt has lt knee osteoarthritus and diabetic retinopthy. Denies pnemonia, but has uncontrolled hypertensn."))\n    assert r["knee osteoarthritis"].match_type == "PHONETIC_FUZZY" and r["knee osteoarthritis"].codes == ["M17.12"]\n    assert r["diabetic retinopathy"].codes == ["E11.319"]\n    assert r["pneumonia"].status == "DO_NOT_CODE"\n    assert r["hypertension"].codes == ["I10"]\n\n\ndef test_transposition_typo(linker):\n    (f,) = linker.process("Patient presents with uncontrolled hypertesnion.")\n    assert f.entity == "hypertension" and f.match_type == "EDIT_FUZZY"\n\n\ndef test_short_word_guardrail(linker):\n    # the thread\'s test failed here: \'hip rain\' vs \'hip pain\' similarity is 0.875\n    assert linker.process("Walked in the hip rain yesterday with no complaints.") == []\n\n\ndef test_allowed_codes_dedup(linker):\n    f = linker.process("HTN. Also hypertension noted.")\n    assert linker.allowed_codes(f) == ["I10"]\n\n\n# ---- ICD-10-CM coding settings and expanded lexicon ---------------------------\n\ndef test_outpatient_vs_inpatient_uncertain(lexicon):\n    from yappu_fsm.clinical import ClinicalLinker\n    note = "Probable pneumonia, start antibiotics."\n    (out,) = ClinicalLinker(lexicon).process(note)\n    (inp,) = ClinicalLinker(lexicon, setting="inpatient").process(note)\n    assert out.status == "DO_NOT_CODE"                      # CM Section IV.H\n    assert inp.codes == ["J18.9"] and inp.assertion == "UNCERTAIN"   # CM Section II.H\n\n\ndef test_inpatient_still_blocks_negation(lexicon):\n    from yappu_fsm.clinical import ClinicalLinker\n    (f,) = ClinicalLinker(lexicon, setting="inpatient").process("Denies fever.")\n    assert f.status == "DO_NOT_CODE"\n\n\ndef test_invalid_setting(lexicon):\n    from yappu_fsm.clinical import ClinicalLinker\n    with pytest.raises(ValueError):\n        ClinicalLinker(lexicon, setting="er")\n\n\n@pytest.mark.parametrize("note,entity,codes", [\n    ("Hx of AFib, on apixaban.", "atrial fibrillation", ["I48.91"]),\n    ("CHF exacerbation.", "heart failure", ["I50.9"]),\n    ("History of CVA in 2020.", "stroke", ["Z86.73"]),\n    ("Father had colon cancer.", "colon cancer", ["Z80.0"]),\n    ("Rt CTS confirmed on EMG.", "carpal tunnel syndrome", ["G56.01"]),\n    ("Bilateral carpal tunnel syndrome.", "carpal tunnel syndrome", ["G56.03"]),\n    ("Left shoulder pain after fall.", "shoulder pain", ["M25.512"]),\n    ("Former smoker, history of nicotine dependence.", "nicotine dependence", ["Z87.891"]),\n    ("UTI, treated with nitrofurantoin.", "urinary tract infection", ["N39.0"]),\n])\ndef test_expanded_cm_lexicon(linker, note, entity, codes):\n    f = by_entity(linker.process(note))[entity]\n    assert f.codes == codes\n\n\n# ---- index ------------------------------------------------------------------------\n\ndef test_index_matches_full_scan(lexicon):\n    import json\n    import random\n\n    from yappu_fsm import data_path\n    from yappu_fsm.clinical import ClinicalLinker\n    fast, slow = ClinicalLinker(lexicon, index=True), ClinicalLinker(lexicon, index=False)\n    notes = [json.loads(l)["note"] for l in open(data_path("gold_assertions.jsonl"), encoding="utf-8")]\n    rng = random.Random(7)\n    for _ in range(300):  # random single-edit typos of lexicon terms inside a sentence\n        term = list(rng.choice(list(lexicon)))\n        i = rng.randrange(len(term))\n        op = rng.choice("sdit")\n        if op == "s":\n            term[i] = rng.choice("abcdefghijklmnopqrstuvwxyz")\n        elif op == "d" and len(term) > 1:\n            del term[i]\n        elif op == "i":\n            term.insert(i, rng.choice("aeiou"))\n        elif op == "t" and i + 1 < len(term):\n            term[i], term[i + 1] = term[i + 1], term[i]\n        notes.append("Patient has " + "".join(term) + " today.")\n    for n in notes:\n        assert [f.to_dict() for f in fast.process(n)] == [f.to_dict() for f in slow.process(n)], n\n\n\ndef test_index_scales(lexicon):\n    import random\n    import string\n    import time\n\n    from yappu_fsm.clinical import ClinicalLinker\n    rng = random.Random(0)\n    big = dict(lexicon)\n    while len(big) < 50_000:\n        big[" ".join("".join(rng.choices(string.ascii_lowercase, k=rng.randint(5, 11)))\n                     for _ in range(rng.randint(1, 4)))] = {"PRESENT_UNSPECIFIED": "Z99.9"}\n    lk = ClinicalLinker(big)\n    note = "Pt has lt knee osteoarthritus and mild NPDR OD. Denies SOB, but has uncontrolled HTN. R/O CAP."\n    lk.process(note)\n    t = time.perf_counter()\n    for _ in range(20):\n        found = lk.process(note)\n    assert (time.perf_counter() - t) / 20 < 0.05\n    assert [f.codes for f in found if f.status == "CODE"] == [["M17.12"], ["E11.3291"], ["I10"]]\n',
    'tests/test_decoder.py': '"""Properties of the pure-Python Venba decoder (no torch needed)."""\nimport random\n\nimport pytest\n\nfrom yappu_fsm.decoding import ICD10CodeTrie, VenbaDecoder\nfrom yappu_fsm.lexicon import VenbaLexicon\nfrom yappu_fsm.prosody import validate_nerisai_venba\n\n\ndef random_poem(dec, rng):\n    s = dec.initial()\n    while True:\n        cands = dec.lex.candidates(s.k, s.expected, s.key_a, s.key_b)\n        assert cands, f"dead end at foot {s.k}"\n        s = dec.advance(s, rng.choice(cands).word.encode())\n        if dec.is_complete(s):\n            return dec.finished_words(s)\n        s = dec.advance(s, b" ")\n        assert s is not None\n\n\n@pytest.mark.parametrize("scheme", ["iru", "oru"])\ndef test_every_decoder_path_is_a_valid_venba(scheme):\n    dec = VenbaDecoder(VenbaLexicon.sample(scheme=scheme))\n    rng = random.Random(42)\n    for _ in range(200):\n        words = random_poem(dec, rng)\n        rep = validate_nerisai_venba(words)\n        assert rep.valid, rep.lines()\n        if scheme == "oru":\n            assert rep.vikarpam == "Oru Vikarpam"\n\n\ndef test_decoder_rejects_invalid_continuations():\n    dec = VenbaDecoder(VenbaLexicon.sample())\n    s = dec.advance(dec.initial(), "எண்ணும்".encode())\n    s = dec.advance(s, b" ")\n    assert dec.advance(s, "வானில்".encode()) is None   # Thema after Thema breaks Vendalai\n    assert dec.advance(s, "எழுத்தும்".encode()) is not None\n    assert dec.advance(s, b" ") is None                   # double space\n\n\ndef test_partial_utf8_bytes_are_accepted():\n    dec = VenbaDecoder(VenbaLexicon.sample())\n    b = "எண்ணும்".encode()\n    s = dec.advance(dec.initial(), b[:2])                 # half a code point\n    assert s is not None\n    assert dec.advance(s, b[2:]) is not None\n\n\ndef test_leading_space_allowed_once():\n    dec = VenbaDecoder(VenbaLexicon.sample())\n    s = dec.advance(dec.initial(), b" ")\n    assert s is not None and dec.advance(s, b" ") is None\n\n\ndef test_icd_trie():\n    t = ICD10CodeTrie(["E11.3291", "E11.3292", "I10"])\n    assert t.is_code("I10") and not t.is_code("I1")\n    assert t.walk("E11.329") is not None and t.walk("E11.33") is None\n',
    'tests/test_hf_integration.py': '"""End-to-end tests through transformers\' model.generate() with a locally trained\nbyte-level BPE tokenizer and a randomly initialised tiny GPT-2 (no downloads)."""\nimport pytest\n\ntorch = pytest.importorskip("torch")\npytest.importorskip("transformers")\npytest.importorskip("tokenizers")\n\nfrom yappu_fsm import data_path  # noqa: E402\nfrom yappu_fsm.clinical import ClinicalLinker, load_lexicon  # noqa: E402\nfrom yappu_fsm.decoding import ICD10LogitsProcessor, VenbaDecoder, VenbaLogitsProcessor, token_bytes_table  # noqa: E402\nfrom yappu_fsm.demo import build_tiny_model, generate_codes, generate_venba  # noqa: E402\nfrom yappu_fsm.lexicon import SAMPLE_WORDS, VenbaLexicon  # noqa: E402\nfrom yappu_fsm.prosody import validate_nerisai_venba  # noqa: E402\n\n\n@pytest.fixture(scope="module")\ndef tiny():\n    lex = load_lexicon(data_path("icd10cm_example_lexicon.json"))\n    codes = [c for v in lex.values() for vv in v.values() for c in ([vv] if isinstance(vv, str) else vv)]\n    corpus = [" ".join(SAMPLE_WORDS)] * 4 + ["CODES: " + "; ".join(codes)] * 4\n    return build_tiny_model(corpus, seed=0)\n\n\ndef test_token_bytes_roundtrip(tiny):\n    tok, _ = tiny\n    table = token_bytes_table(tok)\n    ids = tok("வண்ண நிலவொளி", add_special_tokens=False)["input_ids"]\n    assert b"".join(table[i] for i in ids).decode("utf-8") == "வண்ண நிலவொளி"\n    assert table[tok.eos_token_id] is None\n\n\n@pytest.mark.parametrize("seed", range(4))\ndef test_generate_valid_venba(tiny, seed):\n    tok, model = tiny\n    words, _ = generate_venba(tok, model, seed=seed)\n    rep = validate_nerisai_venba(words)\n    assert len(words) == 15 and rep.valid, rep.lines()\n\n\ndef test_monai_boost_applied(tiny):\n    tok, _ = tiny\n    proc = VenbaLogitsProcessor(tok, VenbaDecoder(VenbaLexicon.sample(), monai_boost=3.0))\n    dec = proc.decoder\n    s = dec.advance(dec.initial(), "வண்ண".encode())\n    s = dec.advance(s, b" ")\n    _, boosted, pozhippu = proc.allowed(s)\n    # foot 1 of line 1: tokens starting a word whose first letter is in the ம/வ + அ group\n    assert boosted and not pozhippu\n\n\ndef test_processor_masks_logits(tiny):\n    tok, _ = tiny\n    proc = VenbaLogitsProcessor(tok, VenbaDecoder(VenbaLexicon.sample()))\n    ids = tok("வெண்பா:", return_tensors="pt")["input_ids"]\n    scores = torch.zeros((1, len(tok)))\n    out = proc(ids, scores.clone())\n    allowed = torch.isfinite(out[0]).nonzero().flatten().tolist()\n    assert allowed and len(allowed) < len(tok)\n    for tid in allowed:\n        assert proc.decoder.advance(proc.decoder.initial(), proc.token_bytes[tid]) is not None\n\n\n@pytest.mark.parametrize("note", [\n    "Pt has lt knee osteoarthritis. Denies SOB, but has uncontrolled HTN. R/O CAP.",\n    "Bilateral hip pain. History of MI.",\n    "Mild NPDR OD and diabetic nefropathy.",\n])\ndef test_generated_codes_subset_of_allowed(tiny, lexicon, note):\n    tok, model = tiny\n    linker = ClinicalLinker(lexicon)\n    allowed = linker.allowed_codes(linker.process(note))\n    assert allowed\n    for seed in range(10):\n        codes, _ = generate_codes(tok, model, allowed, prompt=f"Note: {note}\\nCODES:", seed=seed)\n        assert codes and set(codes) <= set(allowed) and len(codes) == len(set(codes))\n\n\ndef test_icd_processor_ignores_text_before_marker(tiny):\n    tok, _ = tiny\n    proc = ICD10LogitsProcessor(tok, ["I10"])\n    ids = tok("free text", return_tensors="pt")["input_ids"]\n    scores = torch.zeros((1, len(tok)))\n    assert torch.equal(proc(ids, scores.clone()), scores)\n',
    'tests/test_js_port.py': '"""The tutorial\'s JavaScript port must agree with the Python implementation."""\nimport json\nimport pathlib\nimport shutil\nimport subprocess\n\nimport pytest\nfrom conftest import NANNERI, POEM_1, POEM_2\n\nfrom yappu_fsm import encode_asai, ethukai_key, monai_key, split_display, validate_nerisai_venba\nfrom yappu_fsm.lexicon import SAMPLE_WORDS\nfrom yappu_fsm.prosody import eetrucheer_vaipadu\n\nJS = pathlib.Path(__file__).resolve().parents[1] / "docs" / "src" / "prosody.js"\npytestmark = pytest.mark.skipif(shutil.which("node") is None or not JS.exists(), reason="node or docs/src missing")\n\nWORDS = sorted(set(SAMPLE_WORDS) | {"கூவிளங்கனி", "கொக்கு", "ஃபேன்", "தமிழ்", "சங்கம்", "ஔவை"})\nBROKEN = POEM_2.replace("நிலவொளி", "வானில்")\n\n\ndef test_js_matches_python():\n    script = f"""\nconst Y = require({json.dumps(str(JS))});\nconst words = {json.dumps(WORDS, ensure_ascii=False)};\nconst poems = {json.dumps([POEM_1, POEM_2, NANNERI, BROKEN], ensure_ascii=False)};\nconsole.log(JSON.stringify({{\n  w: words.map(w => [Y.encode(w), Y.split(w), Y.ethukai(w), Y.monai(w), Y.eetrucheer(w)]),\n  p: poems.map(p => {{ const r = Y.validate(p); return [r.valid, r.vikarpam, r.feet.map(f => f.thalai)]; }})\n}}));"""\n    out = json.loads(subprocess.run(["node", "-e", script], capture_output=True, text=True, check=True).stdout)\n    for w, js in zip(WORDS, out["w"]):\n        assert js == [encode_asai(w), split_display(w), ethukai_key(w), monai_key(w), eetrucheer_vaipadu(w)], w\n    for p, js in zip([POEM_1, POEM_2, NANNERI, BROKEN], out["p"]):\n        r = validate_nerisai_venba(p)\n        assert js == [r.valid, r.vikarpam, [f.thalai_in for f in r.feet]]\n',
    'tests/test_metrics.py': 'import math\n\nfrom yappu_fsm import data_path\nfrom yappu_fsm.clinical import evaluate, mcc\n\n\ndef binary_mcc(tp, tn, fp, fn):\n    return (tp * tn - fp * fn) / math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))\n\n\ndef test_mcc_binary_matches_formula():\n    y_true = ["1"] * 6 + ["0"] * 4\n    y_pred = ["1"] * 5 + ["0"] + ["0"] * 3 + ["1"]\n    assert abs(mcc(y_true, y_pred) - binary_mcc(tp=5, tn=3, fp=1, fn=1)) < 1e-12\n\n\ndef test_mcc_extremes():\n    assert mcc(list("abcabc"), list("abcabc")) == 1.0\n    assert mcc(list("1100"), list("0011")) == -1.0\n    assert mcc(list("aaaa"), list("aaaa")) == 0.0  # undefined -> 0\n\n\ndef test_gold_set_regression(linker):\n    r = evaluate(linker, data_path("gold_assertions.jsonl"))\n    assert r["assertion_mcc"] >= 0.95\n    assert r["laterality_mcc"] >= 0.99\n    assert r["codable_mcc"] >= 0.90\n',
    'tests/test_negex.py': '"""NegEx / ConText behaviour of the clinical linker."""\nimport pytest\n\nfrom yappu_fsm import data_path\nfrom yappu_fsm.clinical import evaluate\n\n\ndef assertion_of(linker, note, entity):\n    return {f.entity: f.assertion for f in linker.process(note)}[entity]\n\n\n@pytest.mark.parametrize("note,entity,state", [\n    ("Ruled out pneumonia on admission.", "pneumonia", "NEGATED"),                       # [PREN] ruled out\n    ("Pneumonia in the left lower lobe was ruled out.", "pneumonia", "NEGATED"),         # [POST] 5 tokens back\n    ("No history of MI.", "myocardial infarction", "NEGATED"),                           # longest trigger wins\n    ("No fever aside from mild cough.", "cough", "PRESENT"),                             # [CONJ] ends scope\n    ("No fever aside from mild cough.", "fever", "NEGATED"),\n    ("Absence of chest pain at rest.", "chest pain", "NEGATED"),\n    ("Headache was not demonstrated on review.", "headache", "NEGATED"),\n    ("Fever never developed after surgery.", "fever", "NEGATED"),\n    ("Return if fever develops.", "fever", "HYPOTHETICAL"),                              # ConText hypothetical\n    ("If the patient develops chest pain, call 911.", "chest pain", "HYPOTHETICAL"),\n    ("No change in hypertension since last visit.", "hypertension", "PRESENT"),          # [PSEU]\n    ("Pneumonia cannot be ruled out.", "pneumonia", "UNCERTAIN"),\n    ("Not rule out pneumonia yet, start empiric therapy.", "pneumonia", "UNCERTAIN"),\n    ("The patient was evaluated for stroke, which was ruled out.", "stroke", "NEGATED"),\n    ("Cough present; fever absent.", "fever", "NEGATED"),\n    ("Shortness of breath due to heart failure, without chest pain.", "heart failure", "PRESENT"),\n])\ndef test_negex_cases(linker, note, entity, state):\n    assert assertion_of(linker, note, entity) == state\n\n\ndef test_hypothetical_never_coded(linker):\n    (f,) = linker.process("Return if fever develops.")\n    assert f.status == "DO_NOT_CODE"\n\n\ndef test_scope_window_limit(linker):\n    # a pre-trigger reaches at most 6 tokens\n    note = "No acute distress noted by the team during the visit today with hypertension."\n    assert assertion_of(linker, note, "hypertension") == "PRESENT"\n\n\n@pytest.mark.parametrize("gold,floor", [("gold_assertions.jsonl", 0.95), ("gold_negex.jsonl", 0.99),\n                                        ("gold_negex_heldout.jsonl", 0.99)])\ndef test_gold_sets(linker, gold, floor):\n    assert evaluate(linker, data_path(gold))["assertion_mcc"] >= floor\n',
    'tests/test_prosody.py': 'from conftest import NANNERI, POEM_1, POEM_2\n\nfrom yappu_fsm.prosody import eetrucheer_vaipadu, format_venba, parse_venba, thalai_name, validate_nerisai_venba\n\n\ndef test_three_poems_valid():\n    for p in (POEM_1, POEM_2, NANNERI):\n        rep = validate_nerisai_venba(p)\n        assert rep.valid, rep.lines()\n        assert rep.vikarpam == "Iru Vikarpam"\n\n\ndef test_corrected_thalai_labels_poem1():\n    rep = validate_nerisai_venba(POEM_1)\n    labels = {f.word: f.thalai_in for f in rep.feet}\n    assert labels["பண்ணும்"] == "Iyarseer Vendalai"     # after Koovilam கூடினால்\n    assert labels["கண்ணும்"] == "Iyarseer Vendalai"     # after Koovilam பார்ப்பவர்\n    assert labels["நம்மொழி"] == "Iyarseer Vendalai"     # after Karuvilam இலக்கணம்\n    assert labels["கூடினால்"] == "Venseer Vendalai"     # after Kaycheer இசையோடு\n\n\ndef test_kaasu_eetrucheer():\n    assert eetrucheer_vaipadu("கொக்கு") == "Kaasu"\n    assert eetrucheer_vaipadu("பிறப்பு") == "Pirappu"\n    assert eetrucheer_vaipadu("மதி") == "Malar"\n    assert eetrucheer_vaipadu("நாள்") == "Naal"\n    assert eetrucheer_vaipadu("வானில்") is None\n\n\ndef test_thalai_break_detected():\n    words = parse_venba(POEM_2)\n    words[1] = "வானில்"  # Thema after Thema -> Nerondru Aasiriyathalai\n    rep = validate_nerisai_venba(words)\n    assert not rep.valid\n    assert any("Aasiriyathalai" in f.message for f in rep.feet)\n\n\ndef test_kanicheer_rejected():\n    assert thalai_name("110", "1") == "Ondraadha Vanjithalai"\n    words = parse_venba(POEM_2)\n    words[2] = "கூவிளங்கனி"  # 1-0-0 Kanicheer\n    assert not validate_nerisai_venba(words).valid\n\n\ndef test_ethukai_mismatch_and_count():\n    words = parse_venba(POEM_2)\n    words[12] = "நிலவொளி"\n    rep = validate_nerisai_venba(words)\n    assert not rep.valid and any("Ethukai B" in e for e in rep.errors)\n    assert not validate_nerisai_venba(words[:14]).valid\n\n\ndef test_strict_ethukai_rejects_nanneri():\n    assert not validate_nerisai_venba(NANNERI, strict_ethukai=True).valid\n\n\ndef test_format_roundtrip():\n    words = parse_venba(POEM_2)\n    assert parse_venba(format_venba(words)) == words\n',
    'tests/test_rules.py': 'import time\n\nimport pytest\n\nfrom yappu_fsm import data_path\nfrom yappu_fsm.clinical import ClinicalLinker\nfrom yappu_fsm.rules import AceError, DatalogError, RuleCoder, compile_rules, parse_rule, parse_rules\nfrom yappu_fsm.rules.datalog import Literal, Program, Rule, Var\n\n\n@pytest.fixture(scope="module")\ndef coder(lexicon):\n    return RuleCoder(ClinicalLinker(lexicon), data_path("icd10cm_rules.ace"))\n\n\n# ---- controlled-English parser ------------------------------------------------\n\ndef test_parse_and_paraphrase():\n    r = parse_rule("If the patient has a hypertension and the patient has a chronic-kidney-disease "\n                   "then the code I12.9 is assigned and the code I10 is not assigned.", "HTN")\n    assert r.paraphrase() == ("IF { the patient has hypertension } AND { the patient has chronic kidney disease } "\n                              "THEN { code I12.9 is assigned } AND { code I10 is blocked }")\n    texts = [str(d) for d in r.to_datalog()]\n    assert texts[0] == \'code(P, "I12.9") :- patient(P), has(P, hypertension), has(P, "chronic-kidney-disease").\'\n    assert texts[1].startswith(\'blocked(P, "I10")\')\n\n\n@pytest.mark.parametrize("sentence,expected", [\n    ("If the patient has hypertension then the code I10 is assigned.", "\'a\', \'an\', \'the\' or a variable"),\n    ("If the patient has a hypertension then the code I1X0 is assigned.", "an ICD-10-CM code"),\n    ("If the patient had a hypertension then the code I10 is assigned.", "\'has\'"),\n    ("If the patient has a hypertension then the code I10 is assigned", "\'.\'"),\n    ("If the patient has a diagnosis X then the candidate code of Y is assigned.", "a variable introduced earlier"),\n])\ndef test_parse_errors_are_precise(sentence, expected):\n    with pytest.raises(AceError) as e:\n        parse_rule(sentence)\n    assert expected in str(e.value)\n\n\ndef test_unknown_concept_rejected(lexicon):\n    with pytest.raises(AceError) as e:\n        parse_rule("If the patient has a sepsis then the code A41.9 is assigned.", concepts=lexicon.keys())\n    assert "concept from the lexicon" in str(e.value)\n\n\ndef test_all_condition_forms_parse(lexicon):\n    rules = parse_rules("""\n    A: If a patient has a history of a stroke then the code Z86.73 is assigned.\n    B: If a relative of the patient has a colon-cancer then the code Z80.0 is assigned.\n    C: If the patient does not have a fever and the patient has a cough then the code R05.9 is assigned.\n    D: If the encounter is an outpatient-encounter and the patient has an uncertain pneumonia then the code R05.9 is assigned.\n    E: If the patient has a knee-osteoarthritis and the knee-osteoarthritis is on both sides then the code M17.0 precedes the code I10.\n    F: If the code E10.9 is assigned and the code E11.9 is not assigned then the codes E10.9 and E11.9 conflict.\n    """, concepts=lexicon.keys())\n    assert [r.rid for r in rules] == list("ABCDEF")\n    compile_rules(rules)\n\n\ndef test_unstratifiable_rules_rejected():\n    rules = parse_rules("If the code I10 is not assigned then the code I10 is assigned.")\n    with pytest.raises(DatalogError, match="not stratifiable"):\n        compile_rules(rules)\n\n\n# ---- Datalog engine -------------------------------------------------------------\n\ndef test_datalog_recursion_and_negation():\n    X, Y, Z = Var("X"), Var("Y"), Var("Z")\n    prog = Program([\n        Rule(Literal("path", (X, Y)), [Literal("edge", (X, Y))], "r1"),\n        Rule(Literal("path", (X, Z)), [Literal("path", (X, Y)), Literal("edge", (Y, Z))], "r2"),\n        Rule(Literal("cut", (X, Y)), [Literal("node", (X,)), Literal("node", (Y,)), Literal("path", (X, Y), True)], "r3"),\n    ])\n    m = prog.run([("edge", "a", "b"), ("edge", "b", "c"), ("edge", "c", "a")] + [("node", n) for n in "abcd"])\n    assert len(m.query("path")) == 9           # full cycle on a, b, c\n    assert ("cut", "a", "d") in m.facts and ("cut", "a", "b") not in m.facts\n    assert m.explain(("path", "a", "a"))[0].startswith("path(a, a)  [r2]")\n\n\ndef test_unsafe_rule_rejected():\n    X, Y = Var("X"), Var("Y")\n    with pytest.raises(DatalogError, match="unsafe"):\n        Program([Rule(Literal("p", (X, Y)), [Literal("q", (X,))], "bad")])\n\n\n# ---- ICD-10-CM rule set -----------------------------------------------------------\n\ndef test_diabetes_ckd_combination(coder):\n    r = coder.code("Pt with T2DM and CKD.")\n    assert r.codes == ["E11.22", "N18.9"]\n    assert "E11.9" in r.blocked and any("DM_CKD" in line for line in r.proofs["E11.22"])\n\n\ndef test_hypertensive_ckd_and_sequencing(coder):\n    r = coder.code("Pt with T2DM and CKD. Also HTN.")\n    assert r.codes == ["E11.22", "I12.9", "N18.9"]\n    assert set(r.blocked) == {"E11.9", "I10"}\n\n\ndef test_hypertension_alone_unchanged(coder):\n    assert coder.code("Uncontrolled HTN.").codes == ["I10"]\n\n\ndef test_excludes1_conflict_reported(coder):\n    r = coder.code("T1DM noted; also T2DM per old records.")\n    assert [(a, b) for a, b, _ in r.conflicts] == [("E10.9", "E11.9")]\n\n\ndef test_integral_symptoms_suppressed_only_when_diagnosis_coded(coder):\n    confirmed = coder.code("Pneumonia confirmed on imaging. Cough and fever for three days.")\n    assert confirmed.codes == ["J18.9"] and {"R05.9", "R50.9"} <= set(confirmed.blocked)\n    uncertain = coder.code("Probable CAP. Cough and fever for three days.")          # Guideline IV.H\n    assert uncertain.codes == ["R05.9", "R50.9"]\n    inpatient = coder.code("Probable CAP. Cough and fever for three days.", setting="inpatient")  # II.H\n    assert inpatient.codes == ["J18.9"]\n\n\ndef test_known_limitation_trigger_scope_covers_symptoms(coder):\n    # \'probable\' scopes over \'cough and fever\' in the same clause, so they are UNCERTAIN too.\n    r = coder.code("Probable CAP with cough and fever.")\n    assert r.codes == []\n\n\ndef test_inpatient_rule_matches_linker_inpatient_setting(coder, lexicon):\n    """The ACE rule INPT_UNCERTAIN reproduces the hard-coded Guideline II.H behaviour."""\n    import json\n    lk = ClinicalLinker(lexicon, setting="inpatient")\n    for line in open(data_path("gold_assertions.jsonl"), encoding="utf-8"):\n        note = json.loads(line)["note"]\n        expected = set(lk.allowed_codes(lk.process(note)))\n        got = set(coder.code(note, "inpatient").codes) | set(coder.code(note, "inpatient").blocked)\n        assert expected <= got, note\n\n\ndef test_rule_engine_speed(coder):\n    notes = ["Pt with T2DM and CKD. Also HTN.", "Probable CAP. Cough and fever for three days."] * 50\n    t = time.perf_counter()\n    for n in notes:\n        coder.code(n, "inpatient")\n    assert (time.perf_counter() - t) / len(notes) < 0.05       # generous bound for slow CI machines\n\n\ndef test_datalog_export(coder):\n    text = coder.program.to_text()\n    assert "% DM_CKD.1:" in text and \'final(P, C) :- code(P, C), not blocked(P, C).\' in text\n\n\n# ---- sequencing -------------------------------------------------------------------\n\ndef test_derived_code_takes_place_of_replaced_code(coder):\n    # E11.22 replaces E11.9 (1st finding), I12.9 replaces I10 (3rd); N18.9 follows both;\n    # the symptom code goes after the diagnoses\n    r = coder.code("Pt with T2DM and CKD. Also HTN. Probable CAP. Cough for three days.")\n    assert r.codes == ["E11.22", "I12.9", "N18.9", "R05.9"]\n\n\ndef test_symptoms_and_factors_sequenced_last(coder):\n    r = coder.code("Cough for three days. Family history of HTN. Probable CAP. Pt with T2DM and CKD.")\n    assert r.codes == ["E11.22", "N18.9", "R05.9", "Z82.49"]\n    r = coder.code("History of CVA. Headache. Left knee osteoarthritis.")\n    assert r.codes == ["M17.12", "R51.9", "Z86.73"]\n\n\ndef test_inpatient_derived_code_sits_at_its_finding(coder):\n    r = coder.code("Cough for three days. Probable CAP. Fever. HTN.", setting="inpatient")\n    assert r.codes == ["J18.9", "I10"] and {"R05.9", "R50.9"} <= set(r.blocked)\n\n\ndef test_code_variables_and_kinds_parse():\n    r = parse_rule("If a code C is assigned and the code C is a symptom code and a code D is assigned "\n                   "and the code D is a diagnosis code then the code D precedes the code C.")\n    assert r.paraphrase().endswith("THEN { code D is sequenced before code C }")\n    with pytest.raises(AceError, match="only sequencing may use a code variable"):\n        parse_rule("If a code C is assigned then the code C is not assigned.")\n',
    'tests/test_server_cli.py': 'import json\nimport threading\nimport urllib.request\n\nfrom conftest import POEM_2\n\nfrom yappu_fsm.cli import main\nfrom yappu_fsm.server import serve\n\n\ndef test_openai_compatible_endpoint():\n    srv = serve(port=0)\n    port = srv.server_address[1]\n    threading.Thread(target=srv.serve_forever, daemon=True).start()\n    try:\n        def post(body):\n            req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions",\n                                         data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})\n            return json.loads(urllib.request.urlopen(req).read())\n\n        r = post({"model": "yappu-icd-linker", "messages": [{"role": "user", "content": "Denies SOB, but has HTN."}]})\n        assert r["object"] == "chat.completion"\n        content = json.loads(r["choices"][0]["message"]["content"])\n        assert content["allowed_codes"] == ["I10"]\n\n        r = post({"model": "yappu-venba-validator", "messages": [{"role": "user", "content": POEM_2}]})\n        assert "VALID Nerisai Venba" in r["choices"][0]["message"]["content"]\n\n        r = post({"model": "yappu-icd-rules", "setting": "outpatient",\n                  "messages": [{"role": "user", "content": "Pt with T2DM and CKD. Also HTN."}]})\n        assert json.loads(r["choices"][0]["message"]["content"])["codes"] == ["E11.22", "I12.9", "N18.9"]\n\n        models = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models").read())\n        assert {m["id"] for m in models["data"]} == {"yappu-icd-linker", "yappu-icd-rules", "yappu-venba-validator"}\n    finally:\n        srv.shutdown()\n\n\ndef test_cli(capsys):\n    assert main(["asai", "கூடினால்"]) == 0\n    assert "Koovilam" in capsys.readouterr().out\n    assert main(["validate", POEM_2]) == 0\n    assert main(["validate", "எண்ணும் வானில்"]) == 1\n    main(["code", "Pt has lt knee osteoarthritis"])\n    assert "M17.12" in capsys.readouterr().out\n    main(["code", "Pt with T2DM and CKD.", "--rules"])\n    assert "E11.22" in capsys.readouterr().out\n    main(["rules"])\n    assert "rules OK" in capsys.readouterr().out\n    main(["rules", "--datalog"])\n    assert "final(P, C)" in capsys.readouterr().out\n    main(["eval"])\n    assert "assertion_mcc" in capsys.readouterr().out\n',
    'tests/test_tamil.py': 'import pytest\n\nfrom yappu_fsm.tamil import (encode_asai, encode_english, ethukai_key, ethukai_match, graphemes,\n                             has_kutriyalukaram_ending, letter_classes, monai_key, split_display)\n\n\ndef test_graphemes():\n    assert graphemes("வண்ண") == ["வ", "ண்", "ண"]\n    assert graphemes("இருளினை") == ["இ", "ரு", "ளி", "னை"]\n\n\ndef test_decomposed_vowel_signs_are_normalised():\n    decomposed = "கொக்கு"  # கொக்கு with ொ written as ெ + ா\n    assert encode_asai(decomposed) == encode_asai("கொக்கு") == "11"\n\n\ndef test_letter_classes():\n    assert letter_classes("கால்") == ["N", "O"]\n    assert letter_classes("கடாம்") == ["K", "N", "O"]\n\n\n@pytest.mark.parametrize("word,asai", [\n    ("க", "1"), ("கல்", "1"), ("கா", "1"), ("கால்", "1"),\n    ("கட", "0"), ("கடல்", "0"), ("கடா", "0"), ("கடாம்", "0"),\n    ("எண்ணும்", "11"), ("எழுத்தும்", "01"), ("நிலவொளி", "00"), ("இசையோடு", "011"),\n    # words the original PDF / thread split wrongly\n    ("கூடினால்", "10"), ("பார்ப்பவர்", "10"), ("இலக்கணம்", "00"),\n    ("தவழ்ந்திட", "00"), ("ஏற்றிடும்", "10"), ("யிருக்குமாம்", "00"),\n    ("ரறிவிலரென்", "001"), ("விண்மீன்கள்", "111"), ("மதி", "0"),\n])\ndef test_asai(word, asai):\n    assert encode_asai(word) == asai\n\n\ndef test_split_display():\n    assert split_display("கூடினால்") == "கூ/டினால்"\n    assert split_display("இலக்கணம்") == "இலக்/கணம்"\n\n\ndef test_kutriyalukaram():\n    assert has_kutriyalukaram_ending("கொக்கு")\n    assert has_kutriyalukaram_ending("பிறப்பு")\n    assert not has_kutriyalukaram_ending("மதி")\n\n\ndef test_ethukai_consonant_rule():\n    # Nanneri: அடக்க / கடக்கக் / மடைத்தலையில் share ட ; ஓடுமீ / வாடி share ட with nedil first letters\n    assert ethukai_match("அடக்க", "மடைத்தலையில்")\n    assert ethukai_match("ஓடுமீ", "வாடி")\n    assert not ethukai_match("அடக்க", "மடைத்தலையில்", strict=True)\n    # first-letter length must agree\n    assert not ethukai_match("அடக்க", "வாடி")\n    assert ethukai_match("அமுதத்", "நம்மொழி")\n\n\ndef test_monai_ina_ezhuthu():\n    assert monai_key("வண்ண") == monai_key("மண்ணில்")      # ம/வ\n    assert monai_key("தமிழ்") == monai_key("சங்கம்")       # ச/த\n    assert monai_key("இனிய") == monai_key("எண்ணும்")      # இ/எ\n    assert monai_key("கடல்") != monai_key("கிளி")         # different vowel group\n\n\ndef test_english_heuristic_shape():\n    for w in ["moonlight", "melody", "a", "strengths"]:\n        code = encode_english(w)\n        assert set(code) <= {"0", "1"} and 1 <= len(code) <= 3\n',
}

import importlib, json, os, pathlib, subprocess, sys, threading, urllib.request

BASE = pathlib.Path("/content/yappu-fsm" if os.path.isdir("/content") else "yappu-fsm").resolve()
for rel, text in FILES.items():
    p = BASE / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
print(f"wrote {len(FILES)} files to {BASE}")

need = [m for m in ("torch", "transformers", "tokenizers", "pytest") if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
for m in [m for m in sys.modules if m == "yappu_fsm" or m.startswith("yappu_fsm.")]:
    del sys.modules[m]   # pick up fresh code when the cell is re-run

def banner(t): print("\n" + "═" * 78 + f"\n  {t}\n" + "═" * 78)

# 1 ── tests ───────────────────────────────────────────────────────────────────
if RUN_TESTS:
    banner("1 · pytest")
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], cwd=BASE,
                       capture_output=True, text=True, env={**os.environ, "PYTHONPATH": str(BASE)})
    print(r.stdout[-1500:] or r.stderr[-1500:])

# 2 ── prosody ─────────────────────────────────────────────────────────────────
from yappu_fsm import validate_nerisai_venba, split_display, encode_asai
banner("2 · Asai splitter and Nerisai Venba validator")
for w in ["கூடினால்", "இலக்கணம்", "கொக்கு", "தவழ்ந்திட"]:
    print(f"  {w:<12} {split_display(w):<14} {encode_asai(w)}")
POEM = """அடக்க முடையா ரறிவிலரென் றெண்ணிக்
கடக்கக் கருதவும் வேண்டா - மடைத்தலையில்
ஓடுமீ னோட உறுமீன் வருமளவும்
வாடி யிருக்குமாம் கொக்கு."""
print("\n".join(validate_nerisai_venba(POEM).lines()))

# 3 ── clinical linker + MCC ──────────────────────────────────────────────────
from yappu_fsm import data_path
from yappu_fsm.clinical import ClinicalLinker, load_lexicon, evaluate
banner(f"3 · ICD-10-CM linker ({CODING_SETTING})")
LEX = load_lexicon(data_path("icd10cm_example_lexicon.json"))
linker = ClinicalLinker(LEX, setting=CODING_SETTING)
NOTE = ("Pt has lt knee osteoarthritus and mild NPDR OD. Denies SOB, but has uncontrolled HTN. "
        "Probable CAP. Father had colon cancer. History of CVA.")
print("note:", NOTE)
for f in linker.process(NOTE):
    print(f"  {f.raw_mention:<28} → {f.entity:<42} {f.assertion:<15} {f.laterality:<11} "
          f"{','.join(f.codes) or '-':<10} {f.status}")
ev = evaluate(ClinicalLinker(LEX), data_path("gold_assertions.jsonl"))
print({k: v for k, v in ev.items() if k != "errors"})

# 3b ── controlled-English coding rules -> Datalog ─────────────────────────────
banner("3b · ICD-10-CM rules in controlled English, compiled to Datalog")
from yappu_fsm.rules import RuleCoder
coder = RuleCoder(ClinicalLinker(LEX), data_path("icd10cm_rules.ace"))
for r in coder.rules[:2]:
    print(f"{r.rid}: {r.paraphrase()}")
res = coder.code("Pt with T2DM and CKD. Also HTN. Probable CAP. Cough for three days.", CODING_SETTING)
print(res.report())

# 4 ── constrained decoding through model.generate() ──────────────────────────
banner("4 · Constrained decoding (random tiny GPT-2: validity comes from the processor)")
from yappu_fsm.demo import run_demo
run_demo(seed=0)

# 5 ── OpenAI-compatible endpoint ─────────────────────────────────────────────
if RUN_SERVER_DEMO:
    banner("5 · OpenAI-compatible /v1/chat/completions")
    from yappu_fsm.server import serve
    try:
        srv = serve(port=8000)
        threading.Thread(target=srv.serve_forever, daemon=True).start()
    except OSError:
        print("port 8000 already serving (cell re-run) – reusing it")
    body = {"model": "yappu-icd-linker", "setting": CODING_SETTING,
            "messages": [{"role": "user", "content": "Probable pneumonia. Rt CTS. Family history of HTN."}]}
    req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    resp = json.loads(urllib.request.urlopen(req).read())
    print(json.dumps(json.loads(resp["choices"][0]["message"]["content"])["allowed_codes"]))
print("\ndone ✓")
